# S5Mars — pipeline PTQ con ONNX e TensorRT

Questo notebook riparte da zero e costruisce il percorso minimo verso la quantizzazione post-training (PTQ):

1. configurazione essenziale;
2. preprocessing identico al notebook di training;
3. Dataset e DataLoader deterministici;
4. costruzione del modello FP32 e caricamento del checkpoint;
5. baseline PyTorch e ONNX FP32;
6. conversione e benchmark ONNX FP16;
7. PTQ INT8 Entropy sullo scope di operatori scelto in configurazione;
8. esecuzione TensorRT INT8 con fallback FP16;
9. metriche di accuratezza, latenza, energia per immagine e memoria GPU.

La configurazione versionata usa DeepLabV3-ResNet34, calibrazione Entropy random e quantizzazione Conv-only; i bias Conv vengono spostati in Add floating point per evitare DequantizeLinear INT32 incompatibili con TensorRT. Modello e scope INT8 restano configurabili e la copertura QDQ reale viene sempre verificata prima del benchmark.

Il percorso usa `onnx` per rappresentare e controllare il grafo, `onnxruntime` per PTQ ed esecuzione e TensorRT EP per il backend GPU INT8/FP16.

## 1. Dipendenze

La cella mantiene soltanto le librerie necessarie a ricostruire il modello, leggere S5Mars e lavorare con ONNX. `onnxruntime` è separato da `onnx`: il primo esegue il modello e contiene le API PTQ, il secondo gestisce il formato del grafo. Fissiamo `onnxruntime-gpu==1.26.0`, compilato per CUDA 12; le versioni dalla 1.27 in poi richiedono CUDA 13 e non sono compatibili con l'attuale runtime Kaggle.

In [ ]:
# ONNX Runtime CPU e GPU installano lo stesso modulo Python: ne deve restare
# uno solo. La variante GPU include comunque CPUExecutionProvider.
!pip uninstall -y -q onnxruntime onnxruntime-gpu
# ORT 1.26 usa CUDA 12 e il TensorRT EP richiede le librerie TensorRT 10.9.
# Il pacchetto cu12 include runtime, plugin, parser ONNX e binding Python.
!pip install -q onnx onnxscript onnxruntime-gpu==1.26.0 tensorrt-cu12==10.9.0.34 nvidia-ml-py datasets huggingface_hub transformers segmentation-models-pytorch google-api-python-client google-auth


## 2. Configurazione minimale

Sono esposti i parametri che cambiano modello, input, campioni di calibrazione, scope INT8 e gate di esecuzione. La calibrazione usa dati reali ma **non usa le maschere**; le maschere servono soltanto per confrontare FP32 e INT8.

In [ ]:
# Dataset. Il token viene letto solo dalla variabile d'ambiente HF_TOKEN:
# non inserire token o segreti direttamente nel notebook.
REPO_ID = "Mirali33/mb-s5mars"
CALIBRATION_SPLIT = "train"
EVALUATION_SPLIT = "val"
TEST_SPLIT = "test"

# Questi valori devono rimanere identici a quelli usati durante il training.
IMAGE_SIZE = (512, 512)  # (altezza, larghezza)
NUM_CLASSES = 9
IMAGE_MEAN = (0.485, 0.456, 0.406)
IMAGE_STD = (0.229, 0.224, 0.225)

# Il notebook di training corrente usa -100. Poiché le maschere contengono
# ID 0..8, questo significa che nessuna classe viene ignorata. La scelta va
# confermata: tutte le nove classi partecipano alle metriche.
IGNORE_INDEX = -100
CLASS_NAMES = {
    0: "Background",
    1: "Bedrock",
    2: "Hole",
    3: "Ridge",
    4: "Rock",
    5: "Rover",
    6: "Sand / Soil",
    7: "Sky",
    8: "Track",
}

# Target corrente: DeepLabV3-ResNet34 standard, senza oversampling.
# Sono supportati anche SegFormer-B0 e gli altri modelli SMP indicati sotto;
# LCNet verrà aggiunta solo quando diventerà il target corrente.
MODEL_NAME = "smp"  # opzioni: "segformer_b0", "smp"
PRETRAINED_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"
SMP_ARCHITECTURE = "deeplabv3"  # opzioni: unet, deeplabv3, deeplabv3plus
SMP_ENCODER_NAME = "resnet34"  # opzioni già provate: resnet34, mobilenet_v2
# None evita un download inutile: subito dopo caricheremo tutti i pesi dal checkpoint.
SMP_ENCODER_WEIGHTS = None

# ID pubblici delle cartelle degli esperimenti. Le credenziali OAuth restano
# nei Kaggle Secrets e non vengono mai scritte o stampate nel notebook.
DRIVE_CHECKPOINT_FOLDERS = {
    "segformer_b0": "1gor4_Lct_xBMJuWlfxnAXm0fYNjcKBRs",
    "segformer_b0_oversampling_uniform": "17v16dc8D3lNtz5176p1VvMQ1LrEDc6n0",
    "smp_unet_mobilenet_v2": "1Ub1bDNDwNMTm5NSLnyIC6DVm5IRcLCmE",
    "smp_unet_resnet34": "1IJQfHvlQs_g8-uhVWHqWQD0WjkjP6jXg",
    "smp_deeplabv3plus_mobilenet_v2": "15gfIfMeV9Y_KjGjNzA7wizgT4W1GV60o",
    "smp_deeplabv3_resnet34": "113m70R3G5WLJMTLwuUHAOlHT5eLe7lpX",
}
CHECKPOINT_EXPERIMENT = "smp_deeplabv3_resnet34"
# Override opzionale: normalmente resta None e viene usato l'ID della mappa.
CHECKPOINT_FOLDER_ID_OVERRIDE = None
CHECKPOINT_FILENAME = "best.ckpt"  # alternativa: "last.ckpt"
LOCAL_CHECKPOINT_DIR = "/kaggle/working/checkpoints"

# Batch 1 riduce memoria e rende più semplice il primo export/confronto.
BATCH_SIZE = 1
NUM_WORKERS = 2
MAX_CALIBRATION_SAMPLES = 500
CALIBRATION_DRY_RUN_SAMPLES = 3
CALIBRATION_MANIFEST_FILENAME = "int8_calibration_manifest.json"
RUN_INT8_CALIBRATION_PREPARATION = True
MAX_EVALUATION_SAMPLES = None
MAX_TEST_SAMPLES = None  # verifica finale sull'intero test set
# random: campione uniforme e riproducibile.
# class_aware: favorisce la presenza equilibrata delle classi, non i pixel.
CALIBRATION_SELECTION = "random"  # opzioni: "random", "class_aware"
SUBSET_SEED = 42

# FP32 e FP16 useranno CUDA se disponibile. TensorRT non è necessario per
# usare la GPU: lo terremo disattivato finché non avremo validato ONNX CUDA.
# Il fallback CPU viene dichiarato nei log e quindi non passa inosservato.
PREFER_GPU = True
PREFER_TENSORRT = False
TARGET_PRECISIONS = ("fp32", "fp16", "int8")

# False permette di definire e ispezionare tutto senza scaricare dati o pesi.
RUN_SMOKE_TEST = True
RUN_FP32_BASELINE = True

# Benchmark del solo forward: preprocessing e caricamento dati sono esclusi.
LATENCY_WARMUP_RUNS = 10
LATENCY_MEASURED_RUNS = 50
# Energia e memoria vengono misurate in un loop separato e sostenuto:
# il singolo forward è troppo breve rispetto alla granularità di NVML.
EFFICIENCY_WARMUP_RUNS = 10
EFFICIENCY_MIN_RUNS = 100
EFFICIENCY_MIN_SECONDS = 30.0
NVML_SAMPLE_INTERVAL_SECONDS = 0.02

# Export ONNX statico: un solo batch e una sola risoluzione di inferenza.
# Con dynamo=True PyTorch esporta direttamente opset 18. Richiedere 17
# causerebbe un tentativo di conversione non supportato per Resize.
ONNX_OPSET_VERSION = 18
ONNX_OUTPUT_DIR = "/kaggle/working/onnx"
ONNX_FP32_FILENAME = f"{CHECKPOINT_EXPERIMENT}_fp32.onnx"
ONNX_FP16_FILENAME = f"{CHECKPOINT_EXPERIMENT}_fp16.onnx"
RUN_ONNX_EXPORT = True
RUN_ONNX_FP32_BENCHMARK = True
RUN_ONNX_FP16 = True
RUN_ONNX_INT8_QUANTIZATION = True
# Attivare solo dopo avere scelto modello e configurazione sulla validation.
RUN_FINAL_TEST_EVALUATION = False

# Primo esperimento DeepLabV3-ResNet34: Entropy/random e sole Conv in INT8.
# Pooling, Resize, Add e gli altri operatori restano floating point
# (TensorRT li esegue in FP16 quando supportato). I bias INT32 prodotti dal
# default QDQ di ORT generano DequantizeLinear non importabili da TensorRT.
INT8_QUANT_FORMAT = "QDQ"  # opzioni: "QDQ", "QOperator"
INT8_CALIBRATION_METHOD = "Entropy"  # opzioni: MinMax, Entropy, Percentile
INT8_ACTIVATION_TYPE = "QInt8"  # GPU: usare QInt8 (signed)
INT8_WEIGHT_TYPE = "QInt8"
INT8_PER_CHANNEL = True  # una scala per canale di output dei pesi Conv
INT8_REDUCE_RANGE = False  # False usa tutto il range signed a 8 bit
INT8_SYMMETRIC = True  # zero-point 0, richiesto da TensorRT
INT8_OP_TYPES = ("Conv",)
if tuple(INT8_OP_TYPES) == ("Conv",):
    INT8_OPERATOR_SCOPE = "conv_only"
elif set(INT8_OP_TYPES) == {"Conv", "MatMul", "Gemm"}:
    INT8_OPERATOR_SCOPE = "conv_matmul_gemm"
else:
    INT8_OPERATOR_SCOPE = "expanded_ops"
# TensorRT richiede Q/DQ distinti quando uno stesso tensore alimenta più nodi.
INT8_DEDICATED_QDQ_PAIR = True
INT8_QUANTIZE_BIAS = False
# Se il bias resta come terzo input della Conv, il TensorRT EP può ricreare
# internamente un bias_dq INT32 anche con QuantizeBias=False. Lo spostiamo
# quindi in un Add floating point successivo alla Conv nel solo grafo PTQ.
INT8_DECOMPOSE_CONV_BIAS = True
if INT8_DECOMPOSE_CONV_BIAS:
    INT8_BIAS_SCOPE = "conv_bias_add_float"
else:
    INT8_BIAS_SCOPE = "bias_int32" if INT8_QUANTIZE_BIAS else "bias_float"
ONNX_INT8_FILENAME = (
    f"{CHECKPOINT_EXPERIMENT}_int8_qdq_{INT8_CALIBRATION_METHOD.lower()}_"
    f"{CALIBRATION_SELECTION}_{INT8_OPERATOR_SCOPE}_{INT8_BIAS_SCOPE}_trt_fp16.onnx"
)
# Entropy/Percentile espongono molte attivazioni: le elaboriamo in piccoli
# blocchi. 1 usa meno RAM; valori maggiori sono più veloci ma usano più RAM.
# Il valore deve dividere esattamente MAX_CALIBRATION_SAMPLES.
INT8_CALIBRATION_CHUNK_SIZE = 5
# Percentuale di valori mantenuti dal metodo Percentile. Valori più bassi
# tagliano più outlier; 99.999 è il default conservativo di ONNX Runtime.
INT8_CALIBRATION_PERCENTILE = 99.999
# Chiave distinta anche per copertura operatori e fallback TensorRT.
INT8_EXPERIMENT_KEY = (
    f"onnx_int8_{INT8_CALIBRATION_METHOD.lower()}_{CALIBRATION_SELECTION}"
    f"_{INT8_OPERATOR_SCOPE}_{INT8_BIAS_SCOPE}_trt_fp16"
)

# Primo gate TensorRT: crea la sessione e fa un solo forward, senza benchmark.
RUN_TRT_INT8_PROBE = True
RUN_ONNX_INT8_BENCHMARK = True
TRT_DEVICE_ID = 0
TRT_INT8_ENABLE = True
TRT_FP16_ENABLE = True  # i nodi TensorRT non INT8 possono usare FP16
TRT_ENGINE_CACHE_DIR = "/kaggle/working/onnx/trt_int8_engine_cache"
TRT_TIMING_CACHE_DIR = "/kaggle/working/onnx/trt_timing_cache"

# True mantiene input e output FP32 e converte in FP16 il calcolo interno.
# È il default più semplice: dataset e chiamate ORT non cambiano. Con False
# anche input/output diventano FP16 e il chiamante deve fornire float16.
FP16_KEEP_IO_TYPES = True


## 3. Import e riproducibilità

In [ ]:
import ctypes
import gc
import importlib.util
import io
import json
import logging
import os
import random
import time
import threading
from pathlib import Path

import numpy as np
import onnx

# torch va importato prima di ONNX Runtime: così le librerie CUDA e cuDNN
# incluse nell'ambiente sono già caricate quando ORT crea il provider GPU.
import torch


def preload_tensorrt_shared_libraries():
    """Carica le tre librerie native che ONNX Runtime TensorRT richiede."""
    spec = importlib.util.find_spec("tensorrt_libs")
    if spec is None or not spec.submodule_search_locations:
        raise RuntimeError(
            "tensorrt_libs non trovato: esegui prima la cella Dipendenze"
        )
    library_dir = Path(next(iter(spec.submodule_search_locations)))
    required_sonames = (
        "libnvinfer.so.10",
        "libnvinfer_plugin.so.10",
        "libnvonnxparser.so.10",
    )
    handles = []
    loaded_paths = []
    for soname in required_sonames:
        # Preferiamo il symlink con il SONAME esatto; il glob copre anche
        # wheel che contengono soltanto il nome completo con patch version.
        candidates = sorted(
            library_dir.glob(f"{soname}*"), key=lambda path: len(path.name)
        )
        if not candidates:
            raise RuntimeError(f"Libreria TensorRT mancante: {soname}")
        selected = candidates[0]
        handles.append(
            ctypes.CDLL(str(selected), mode=getattr(ctypes, "RTLD_GLOBAL", 0))
        )
        loaded_paths.append(str(selected))
    return library_dir, handles, loaded_paths


# Importare il modulo Python non basta sempre a rendere visibili le librerie
# native al loader di ORT. Conserviamo gli handle fino a fine processo.
TENSORRT_LIBRARY_HANDLES = []
TENSORRT_LOADED_PATHS = []
if RUN_TRT_INT8_PROBE:
    TENSORRT_LIBRARY_DIR, TENSORRT_LIBRARY_HANDLES, TENSORRT_LOADED_PATHS = (
        preload_tensorrt_shared_libraries()
    )
    import tensorrt as trt

import onnxruntime as ort
import pynvml
import segmentation_models_pytorch as smp
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build as build_google_api
from googleapiclient.http import MediaIoBaseDownload
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import SegformerForSemanticSegmentation
from onnxruntime.transformers.float16 import convert_float_to_float16

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("s5mars_onnx_ptq")


def set_seed(seed=42):
    """Rende ripetibile l'ordine dei campioni e le operazioni PyTorch di base."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)
print("torch", torch.__version__)
print("torch CUDA build", torch.version.cuda)
print("torch CUDA available", torch.cuda.is_available())
print("onnx", onnx.__version__)
print("onnxruntime", ort.__version__)
if RUN_TRT_INT8_PROBE:
    print("TensorRT", trt.__version__)
    print("TensorRT native libraries", TENSORRT_LOADED_PATHS)
print("available ORT providers", ort.get_available_providers())


def get_execution_providers(
    prefer_gpu=PREFER_GPU, prefer_tensorrt=PREFER_TENSORRT
):
    """Costruisce l'ordine TensorRT -> CUDA -> CPU e segnala i fallback."""
    available = ort.get_available_providers()
    providers = []
    if prefer_gpu and prefer_tensorrt:
        if "TensorrtExecutionProvider" in available:
            providers.append("TensorrtExecutionProvider")
        else:
            logger.warning("TensorrtExecutionProvider non disponibile")
    if prefer_gpu and "CUDAExecutionProvider" in available:
        providers.append("CUDAExecutionProvider")
    if prefer_gpu and "CUDAExecutionProvider" not in available:
        logger.warning("CUDAExecutionProvider non disponibile: verrà usata la CPU")
    providers.append("CPUExecutionProvider")
    return providers


ORT_PROVIDERS = get_execution_providers()
TORCH_DEVICE = torch.device(
    "cuda" if PREFER_GPU and torch.cuda.is_available() else "cpu"
)
print("torch device", TORCH_DEVICE)
print("ORT provider priority", ORT_PROVIDERS)

# artifacts descrive i file reali; benchmarks conserva metriche e latenze.
# In questo modo lo stesso artefatto può essere provato con runtime diversi.
BENCHMARK_RESULTS = {
    "experiment": {
        "model": CHECKPOINT_EXPERIMENT,
        "input_shape": [1, 3, *IMAGE_SIZE],
        "evaluation_split": EVALUATION_SPLIT,
        "ignore_index": IGNORE_INDEX,
    },
    "artifacts": {},
    "benchmarks": {},
    "calibration": {},
}


def save_benchmark_results():
    """Salva il dizionario anche su JSON, così sopravvive al notebook."""
    output_path = Path(ONNX_OUTPUT_DIR) / "benchmark_results.json"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(BENCHMARK_RESULTS, indent=2), encoding="utf-8"
    )
    return output_path


## 4. Checkpoint da Google Drive tramite Kaggle Secrets

Le credenziali attese sono GDRIVE_CLIENT_ID, GDRIVE_CLIENT_SECRET e GDRIVE_REFRESH_TOKEN. Il codice cerca prima una variabile d'ambiente, utile fuori da Kaggle, e poi il Secret omonimo. Nessun valore viene stampato. La cartella dell'esperimento contiene il file best.ckpt o last.ckpt.

In [ ]:
def get_secret(name, env_name=None, required=True):
    """Legge un segreto senza inserirlo nel notebook o mostrarlo nei log."""
    value = os.environ.get(env_name or name)
    if not value:
        try:
            from kaggle_secrets import UserSecretsClient

            value = UserSecretsClient().get_secret(name)
        except Exception:
            value = None

    if required and not value:
        raise RuntimeError(f"Secret non disponibile: {name}")
    return value


def build_drive_service():
    """Crea un client Drive usato dal notebook soltanto per leggere file."""
    credentials = Credentials(
        token=None,
        refresh_token=get_secret("GDRIVE_REFRESH_TOKEN"),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=get_secret("GDRIVE_CLIENT_ID"),
        client_secret=get_secret("GDRIVE_CLIENT_SECRET"),
    )
    return build_google_api("drive", "v3", credentials=credentials, cache_discovery=False)


def resolve_checkpoint_folder_id():
    folder_id = CHECKPOINT_FOLDER_ID_OVERRIDE or DRIVE_CHECKPOINT_FOLDERS.get(
        CHECKPOINT_EXPERIMENT
    )
    if not folder_id:
        raise ValueError(
            f"Manca l'ID Drive per l'esperimento {CHECKPOINT_EXPERIMENT!r}. "
            "Imposta CHECKPOINT_FOLDER_ID_OVERRIDE prima di continuare."
        )
    # Gli ID Drive contengono solo caratteri alfanumerici, '-' e '_'.
    if not folder_id.replace("-", "").replace("_", "").isalnum():
        raise ValueError("Formato non valido per CHECKPOINT_FOLDER_ID_OVERRIDE")
    return folder_id


def download_checkpoint_from_drive(force=False):
    """Scarica un checkpoint preciso dalla cartella Drive selezionata."""
    destination_dir = Path(LOCAL_CHECKPOINT_DIR)
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination = destination_dir / f"{CHECKPOINT_EXPERIMENT}__{CHECKPOINT_FILENAME}"
    if destination.is_file() and not force:
        logger.info("Checkpoint locale già presente: %s", destination)
        return destination

    folder_id = resolve_checkpoint_folder_id()
    service = build_drive_service()
    response = service.files().list(
        q=f"'{folder_id}' in parents and trashed = false",
        fields="files(id,name,size,modifiedTime)",
        pageSize=1000,
    ).execute()
    matches = [item for item in response.get("files", []) if item["name"] == CHECKPOINT_FILENAME]
    if len(matches) != 1:
        available_names = sorted(item["name"] for item in response.get("files", []))
        raise FileNotFoundError(
            f"Atteso un solo {CHECKPOINT_FILENAME!r}, trovati {len(matches)}. "
            f"File disponibili: {available_names}"
        )

    request = service.files().get_media(fileId=matches[0]["id"])
    with io.FileIO(destination, "wb") as output_file:
        downloader = MediaIoBaseDownload(output_file, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status is not None:
                logger.info("Download checkpoint: %.0f%%", status.progress() * 100)

    logger.info("Checkpoint scaricato in %s", destination)
    return destination


## 5. Preprocessing, Dataset e DataLoader

Il resize e la normalizzazione replicano il training. È importante non cambiare preprocessing durante calibrazione o valutazione: cambierebbe la distribuzione degli input e renderebbe il confronto non valido. Il loader non usa shuffle, così gli esperimenti PTQ restano ripetibili.

In [ ]:
class SegmentationTransform:
    """Converte una coppia PIL (RGB, mask) nei tensori usati dal training."""

    def __init__(self, size, mean, std):
        self.size = tuple(size)
        self.mean = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(3, 1, 1)

    def __call__(self, image, mask):
        # BILINEAR è adatto alle immagini continue; NEAREST evita di creare
        # ID di classe inesistenti nella maschera durante il resize.
        width, height = self.size[1], self.size[0]
        image = image.resize((width, height), Image.Resampling.BILINEAR)
        mask = mask.resize((width, height), Image.Resampling.NEAREST)

        # HWC uint8 [0,255] -> CHW float32 [0,1] -> normalizzazione ImageNet.
        image_array = np.asarray(image, dtype=np.float32) / 255.0
        image_tensor = torch.from_numpy(image_array).permute(2, 0, 1).contiguous()
        image_tensor = (image_tensor - self.mean) / self.std

        # Le maschere restano interi [H,W]; non vengono normalizzate.
        mask_array = np.asarray(mask, dtype=np.int64)
        if mask_array.ndim == 3:
            mask_array = mask_array[..., 0]
        mask_tensor = torch.from_numpy(mask_array).long()
        return image_tensor.float(), mask_tensor


def select_dataset_subset(dataset, max_samples, strategy="random", seed=42):
    """Seleziona un sottoinsieme riproducibile senza decodificare tutte le mask."""
    if max_samples is None or int(max_samples) >= len(dataset):
        return dataset

    sample_count = int(max_samples)
    if sample_count <= 0:
        raise ValueError("max_samples must be positive or None")

    rng = np.random.default_rng(seed)
    shuffled_indices = rng.permutation(len(dataset)).tolist()
    if strategy == "random":
        return dataset.select(shuffled_indices[:sample_count])
    if strategy != "class_aware":
        raise ValueError("strategy must be 'random' or 'class_aware'")

    # class_labels elenca le classi presenti in ogni immagine. È molto più
    # economico delle mask, ma non informa su quanti pixel appartengono a una classe.
    labels_by_index = [set(labels) for labels in dataset["class_labels"]]
    all_labels = sorted(set().union(*labels_by_index))
    pools = {
        label: [index for index in shuffled_indices if label in labels_by_index[index]]
        for label in all_labels
    }
    positions = {label: 0 for label in all_labels}
    presence_counts = {label: 0 for label in all_labels}
    selected = []
    selected_set = set()

    while len(selected) < sample_count:
        available_labels = []
        for label in all_labels:
            pool = pools[label]
            while positions[label] < len(pool) and pool[positions[label]] in selected_set:
                positions[label] += 1
            if positions[label] < len(pool):
                available_labels.append(label)
        if not available_labels:
            break

        # A ogni passo scegliamo la classe finora meno rappresentata.
        target_label = min(available_labels, key=lambda label: (presence_counts[label], label))
        index = pools[target_label][positions[target_label]]
        positions[target_label] += 1
        selected.append(index)
        selected_set.add(index)
        for label in labels_by_index[index]:
            presence_counts[label] += 1

    # Completa da una permutazione uniforme se qualche esempio non ha label note.
    if len(selected) < sample_count:
        selected.extend(
            index for index in shuffled_indices if index not in selected_set
        )
        selected = selected[:sample_count]

    logger.info("Class-aware calibration presence counts: %s", presence_counts)
    return dataset.select(selected)


class S5MarsHFDataset(Dataset):
    """Adattatore minimale tra il dataset Hugging Face e PyTorch."""

    def __init__(self, repo_id, split, transform, token=None, max_samples=None, selection="random", seed=42):
        logger.info("Loading dataset repo=%s split=%s", repo_id, split)
        self.dataset = load_dataset(repo_id, split=split, token=token)
        self.transform = transform
        # Conserviamo fingerprint e posizione nel dataset originale prima
        # della selezione random/class-aware. Serviranno nel manifesto PTQ.
        self.source_fingerprint = self.dataset._fingerprint
        self.source_sample_count = len(self.dataset)

        required_columns = {"image", "mask", "class_labels"}
        missing = required_columns - set(self.dataset.column_names)
        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")

        self.dataset = self.dataset.add_column(
            "_source_index", list(range(len(self.dataset)))
        )
        self.dataset = select_dataset_subset(
            self.dataset, max_samples=max_samples, strategy=selection, seed=seed
        )
        self.subset_fingerprint = self.dataset._fingerprint
        self.selected_source_indices = [
            int(value) for value in self.dataset["_source_index"]
        ]

        logger.info("Loaded %d samples", len(self.dataset))

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = self.dataset[index]
        image = sample["image"].convert("RGB")
        mask = sample["mask"].convert("L")
        image_tensor, mask_tensor = self.transform(image, mask)
        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "index": int(index),
            "source_index": int(sample["_source_index"]),
        }


def segmentation_collate_fn(batch):
    """Raggruppa i campioni mantenendo nomi e shape leggibili."""
    return {
        "image": torch.stack([sample["image"] for sample in batch]),
        "mask": torch.stack([sample["mask"] for sample in batch]),
        "index": torch.tensor([sample["index"] for sample in batch], dtype=torch.long),
        "source_index": torch.tensor(
            [sample["source_index"] for sample in batch], dtype=torch.long
        ),
    }


def build_dataset(split, max_samples=None, selection="random"):
    transform = SegmentationTransform(IMAGE_SIZE, IMAGE_MEAN, IMAGE_STD)
    token = get_secret("hf_token", env_name="HF_TOKEN", required=False)
    return S5MarsHFDataset(
        repo_id=REPO_ID,
        split=split,
        token=token,
        transform=transform,
        max_samples=max_samples,
        selection=selection,
        seed=SUBSET_SEED,
    )


def build_dataloader(dataset, batch_size=BATCH_SIZE):
    # shuffle=False è intenzionale: calibrazione e benchmark devono vedere
    # gli stessi campioni nello stesso ordine a ogni esecuzione.
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=TORCH_DEVICE.type == "cuda",
        drop_last=False,
        collate_fn=segmentation_collate_fn,
    )


def build_calibration_and_evaluation_loaders():
    calibration_dataset = build_dataset(
        CALIBRATION_SPLIT, MAX_CALIBRATION_SAMPLES, CALIBRATION_SELECTION
    )
    evaluation_dataset = build_dataset(EVALUATION_SPLIT, MAX_EVALUATION_SAMPLES)
    return (
        build_dataloader(calibration_dataset),
        build_dataloader(evaluation_dataset),
    )


## 6. Modello FP32 e checkpoint

Queste classi mantengono lo stesso contratto del training: input `[B,3,H,W]`, output logits `[B,9,H,W]`. Non contengono freeze, loss, optimizer o scheduler perché la PTQ non aggiorna i pesi.

In [ ]:
class SegFormerB0ForMars(nn.Module):
    """Wrapper identico al training, ridotto alla sola inferenza."""

    def __init__(self, pretrained_name, num_classes, ignore_index):
        super().__init__()
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            pretrained_name,
            num_labels=num_classes,
            semantic_loss_ignore_index=ignore_index,
            ignore_mismatched_sizes=True,
        )

    def forward(self, images):
        input_hw = images.shape[-2:]
        raw_logits = self.model(pixel_values=images).logits
        return F.interpolate(raw_logits, size=input_hw, mode="bilinear", align_corners=False)


class SMPModelForMars(nn.Module):
    """Wrapper minimale per U-Net, DeepLabV3 e DeepLabV3+ di SMP."""

    def __init__(self, architecture, encoder_name, encoder_weights, num_classes):
        super().__init__()
        constructors = {
            "unet": smp.Unet,
            "deeplabv3": smp.DeepLabV3,
            "deeplabv3plus": smp.DeepLabV3Plus,
        }
        if architecture not in constructors:
            raise ValueError(f"Unsupported SMP architecture: {architecture}")
        self.model = constructors[architecture](
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=num_classes,
            activation=None,
        )

    def forward(self, images):
        return self.model(images)


def build_model():
    """Crea esattamente l'architettura sulla quale caricare il checkpoint."""
    if MODEL_NAME == "segformer_b0":
        return SegFormerB0ForMars(PRETRAINED_NAME, NUM_CLASSES, IGNORE_INDEX)
    if MODEL_NAME == "smp":
        return SMPModelForMars(
            SMP_ARCHITECTURE,
            SMP_ENCODER_NAME,
            SMP_ENCODER_WEIGHTS,
            NUM_CLASSES,
        )
    raise ValueError("MODEL_NAME must be 'segformer_b0' or 'smp'")


def validate_checkpoint_config(checkpoint_config):
    """Blocca il caricamento se checkpoint e architettura non coincidono."""
    if not checkpoint_config:
        logger.warning("Il checkpoint non contiene config: compatibilità non verificabile")
        return

    expected = {"model_name": MODEL_NAME}
    if MODEL_NAME == "smp":
        expected.update(
            smp_architecture=SMP_ARCHITECTURE,
            smp_encoder_name=SMP_ENCODER_NAME,
        )

    mismatches = {
        key: {"expected": expected_value, "checkpoint": checkpoint_config.get(key)}
        for key, expected_value in expected.items()
        if checkpoint_config.get(key) not in (None, expected_value)
    }
    if mismatches:
        raise ValueError(f"Checkpoint incompatibile con la configurazione: {mismatches}")


def load_trained_model(checkpoint_path):
    """Carica su CPU i pesi prodotti dal notebook di training."""
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    model = build_model()
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    if isinstance(checkpoint, dict):
        validate_checkpoint_config(checkpoint.get("config"))
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state_dict, strict=True)

    # eval() è obbligatorio: BatchNorm e Dropout devono comportarsi come in inferenza.
    model.eval()
    return model


## 7. Smoke test opzionale

Questa cella controlla il contratto minimo prima dell'export: dati `float32`, shape coerenti, checkpoint caricabile e output `[B,9,H,W]`. Non misura ancora accuratezza o prestazioni.

In [ ]:
if RUN_SMOKE_TEST:
    checkpoint_path = download_checkpoint_from_drive()
    model = load_trained_model(checkpoint_path).to(TORCH_DEVICE)
    calibration_dataset = build_dataset(
        CALIBRATION_SPLIT, MAX_CALIBRATION_SAMPLES, CALIBRATION_SELECTION
    )
    calibration_loader = build_dataloader(calibration_dataset)
    batch = next(iter(calibration_loader))
    images = batch["image"].to(TORCH_DEVICE, non_blocking=True)

    with torch.inference_mode():
        logits = model(images)

    assert batch["image"].dtype == torch.float32
    assert tuple(logits.shape[:2]) == (batch["image"].shape[0], NUM_CLASSES)
    assert logits.shape[-2:] == batch["image"].shape[-2:]

    print("images:", tuple(batch["image"].shape), batch["image"].dtype)
    print("masks: ", tuple(batch["mask"].shape), batch["mask"].dtype)
    print("logits:", tuple(logits.shape), logits.dtype)
else:
    print("Base pronta. Imposta RUN_SMOKE_TEST=True per scaricare checkpoint e dati.")


## 8. Baseline FP32: accuratezza di riferimento

Valutiamo il checkpoint sullo split val prima di modificare il modello. Le metriche non dipendono dal fatto che PyTorch stia usando CPU o GPU. Misuriamo anche una latenza PyTorch preliminare, sempre etichettata con il dispositivo effettivo; il confronto finale userà invece lo stesso backend ONNX Runtime/TensorRT per tutte le precisioni.

La matrice di confusione accumula, per ogni pixel valido, classe reale sulle righe e classe predetta sulle colonne. Da questa matrice ricaviamo pixel accuracy, mIoU e IoU per classe.

In [ ]:
def update_confusion_matrix(confusion_matrix, predictions, targets):
    """Aggiunge un batch alla matrice di confusione globale."""
    predictions = predictions.reshape(-1)
    targets = targets.reshape(-1)

    # -100 non compare nelle mask S5Mars, quindi con la configurazione corrente
    # tutte le classi 0..8 vengono incluse. Il controllo resta generico.
    valid = (targets != IGNORE_INDEX) & (targets >= 0) & (targets < NUM_CLASSES)
    predictions = predictions[valid]
    targets = targets[valid]

    flat_indices = targets * NUM_CLASSES + predictions
    batch_matrix = torch.bincount(
        flat_indices, minlength=NUM_CLASSES * NUM_CLASSES
    ).reshape(NUM_CLASSES, NUM_CLASSES)
    confusion_matrix += batch_matrix.cpu().to(torch.float64)


def metrics_from_confusion_matrix(confusion_matrix):
    """Calcola le metriche senza conservare logits o mask in memoria."""
    true_positive = torch.diag(confusion_matrix)
    support = confusion_matrix.sum(dim=1)
    predicted = confusion_matrix.sum(dim=0)
    union = support + predicted - true_positive

    iou = torch.full((NUM_CLASSES,), float("nan"), dtype=torch.float64)
    present = union > 0
    iou[present] = true_positive[present] / union[present]

    # La media usa soltanto classi realmente presenti nel ground truth.
    valid_classes = support > 0
    if 0 <= IGNORE_INDEX < NUM_CLASSES:
        valid_classes[IGNORE_INDEX] = False
    mean_iou = iou[valid_classes].mean().item()
    pixel_accuracy = (
        true_positive.sum() / confusion_matrix.sum().clamp(min=1.0)
    ).item()

    per_class_iou = {
        CLASS_NAMES[class_id]: (iou[class_id].item() if present[class_id] else None)
        for class_id in range(NUM_CLASSES)
    }
    return {
        "pixel_accuracy": pixel_accuracy,
        "miou": mean_iou,
        "per_class_iou": per_class_iou,
    }


def synchronize_torch_device(device):
    """Attende la GPU; su CPU non serve alcuna sincronizzazione."""
    if device.type == "cuda":
        torch.cuda.synchronize(device)


@torch.inference_mode()
def measure_pytorch_latency(model, sample, device):
    """Misura il solo forward su un input già pronto, con batch 1."""
    if sample.shape[0] != 1:
        raise ValueError("Il benchmark di latenza richiede batch size 1")

    sample = sample.to(device, non_blocking=True)
    model.eval()

    # Il warm-up assorbe inizializzazioni e cache che falserebbero le prime run.
    for _ in range(LATENCY_WARMUP_RUNS):
        model(sample)
    synchronize_torch_device(device)

    latencies_ms = []
    for _ in range(LATENCY_MEASURED_RUNS):
        synchronize_torch_device(device)
        start = time.perf_counter()
        model(sample)
        synchronize_torch_device(device)
        latencies_ms.append((time.perf_counter() - start) * 1000.0)

    mean_ms = float(np.mean(latencies_ms))
    return {
        "device": str(device),
        "batch_size": 1,
        "warmup_runs": LATENCY_WARMUP_RUNS,
        "measured_runs": LATENCY_MEASURED_RUNS,
        "measurement_scope": "forward con input già sul device",
        "mean_ms": mean_ms,
        "median_ms": float(np.median(latencies_ms)),
        "p95_ms": float(np.percentile(latencies_ms, 95)),
        "images_per_second": 1000.0 / mean_ms,
    }


@torch.inference_mode()
def evaluate_pytorch_fp32(model, dataloader, device, split_name=None):
    """Esegue inferenza FP32 e restituisce solo metriche di accuratezza."""
    model.eval()
    confusion_matrix = torch.zeros(
        (NUM_CLASSES, NUM_CLASSES), dtype=torch.float64
    )

    for batch_number, batch in enumerate(dataloader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        targets = batch["mask"].to(device, non_blocking=True)
        predictions = model(images).argmax(dim=1)
        update_confusion_matrix(confusion_matrix, predictions, targets)

        if batch_number % 25 == 0 or batch_number == len(dataloader):
            logger.info("FP32 evaluation: %d/%d batch", batch_number, len(dataloader))

    metrics = metrics_from_confusion_matrix(confusion_matrix)
    metrics["split"] = split_name or EVALUATION_SPLIT
    metrics["samples"] = len(dataloader.dataset)
    return metrics


if RUN_FP32_BASELINE:
    if "model" not in globals():
        raise RuntimeError("Esegui prima la cella dello smoke test")

    evaluation_dataset = build_dataset(
        EVALUATION_SPLIT, MAX_EVALUATION_SAMPLES
    )
    evaluation_loader = build_dataloader(evaluation_dataset)
    fp32_reference_metrics = evaluate_pytorch_fp32(
        model, evaluation_loader, TORCH_DEVICE
    )
    latency_sample = next(iter(evaluation_loader))["image"]
    pytorch_fp32_result = {
        "runtime": "pytorch",
        "precision": "fp32",
        "device": str(TORCH_DEVICE),
        "artifact_key": "pytorch_fp32",
        "metrics": fp32_reference_metrics,
        "latency": measure_pytorch_latency(
            model, latency_sample, TORCH_DEVICE
        ),
    }
    BENCHMARK_RESULTS["artifacts"]["pytorch_fp32"] = {
        "format": "pytorch_checkpoint",
        "precision": "fp32",
        "path": str(checkpoint_path),
        "size_mib": Path(checkpoint_path).stat().st_size / (1024 ** 2),
        "source": "Google Drive checkpoint",
    }
    BENCHMARK_RESULTS["benchmarks"]["pytorch_fp32_cuda"] = pytorch_fp32_result
    results_path = save_benchmark_results()
    print(json.dumps(pytorch_fp32_result, indent=2))
    print("Risultati cumulativi salvati in:", results_path)


## 9. Export ONNX FP32 statico

Esportiamo il checkpoint FP32 in un singolo file ONNX. L'input e l'output hanno shape fissa: batch 1, tre canali e risoluzione 512x512. Non impostiamo assi dinamici perché il target di inferenza è già noto e i provider GPU possono ottimizzare meglio shape statiche.

Usiamo direttamente opset 18, quello prodotto dal nuovo exporter PyTorch per questa rete: evitiamo così il tentativo fallito di convertire l'operazione Resize a opset 17. dynamo=True abilita l'exporter basato su torch.export e richiede onnxscript. external_data=False conserva grafo e pesi nello stesso file.

In [ ]:
def onnx_tensor_shape(value_info):
    """Converte la shape protobuf ONNX in una tupla Python leggibile."""
    dimensions = value_info.type.tensor_type.shape.dim
    return tuple(
        dimension.dim_value if dimension.HasField("dim_value") else None
        for dimension in dimensions
    )


def export_onnx_fp32(model):
    """Esporta il modello FP32 e controlla struttura, nomi e shape."""
    output_dir = Path(ONNX_OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / ONNX_FP32_FILENAME

    model.eval()
    parameter = next(model.parameters())
    if parameter.dtype != torch.float32:
        raise ValueError(f"Atteso modello FP32, trovato {parameter.dtype}")

    # Il contenuto del tensore non influenza il grafo; servono dtype e shape.
    example_input = torch.zeros(
        (1, 3, *IMAGE_SIZE), dtype=torch.float32, device=parameter.device
    )

    torch.onnx.export(
        model,
        (example_input,),
        str(output_path),
        input_names=["images"],
        output_names=["logits"],
        opset_version=ONNX_OPSET_VERSION,
        dynamo=True,
        external_data=False,
        export_params=True,
        keep_initializers_as_inputs=False,
    )

    # checker verifica che il file rispetti lo schema ONNX. Non confronta
    # ancora i valori numerici: quella sarà la fase di parità PyTorch/ORT.
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model, full_check=True)

    if len(onnx_model.graph.input) != 1 or len(onnx_model.graph.output) != 1:
        raise ValueError("Il modello ONNX deve avere un input e un output")

    input_info = onnx_model.graph.input[0]
    output_info = onnx_model.graph.output[0]
    input_shape = onnx_tensor_shape(input_info)
    output_shape = onnx_tensor_shape(output_info)
    expected_input_shape = (1, 3, *IMAGE_SIZE)
    expected_output_shape = (1, NUM_CLASSES, *IMAGE_SIZE)

    if input_info.name != "images" or input_shape != expected_input_shape:
        raise ValueError(f"Input ONNX inatteso: {input_info.name}, {input_shape}")
    if output_info.name != "logits" or output_shape != expected_output_shape:
        raise ValueError(f"Output ONNX inatteso: {output_info.name}, {output_shape}")

    default_opset = next(
        item.version for item in onnx_model.opset_import if item.domain in ("", "ai.onnx")
    )
    return output_path, {
        "path": str(output_path),
        "size_mib": output_path.stat().st_size / (1024 ** 2),
        "opset": default_opset,
        "input_name": input_info.name,
        "input_shape": input_shape,
        "output_name": output_info.name,
        "output_shape": output_shape,
        "onnx_checker": "passed",
    }


if RUN_ONNX_EXPORT:
    if "model" not in globals():
        raise RuntimeError("Esegui prima lo smoke test e carica il checkpoint")
    onnx_fp32_path, onnx_export_info = export_onnx_fp32(model)
    BENCHMARK_RESULTS["artifacts"]["onnx_fp32"] = {
        "format": "onnx",
        "precision": "fp32",
        **onnx_export_info,
    }
    save_benchmark_results()
    print(json.dumps(onnx_export_info, indent=2))


## 10. Verifica e benchmark ONNX Runtime FP32

Prima di ridurre la precisione verifichiamo la baseline ONNX FP32. Il controllo numerico confronta gli stessi logits prodotti da PyTorch e ONNX Runtime; la prediction agreement confronta invece la classe scelta per ogni pixel.

La latenza esclude preprocessing e caricamento dati. Quando il provider principale è CUDA o TensorRT usiamo I/O Binding: l'input viene copiato sulla GPU una sola volta prima del cronometro e l'output resta sulla GPU, rendendo la misura confrontabile con il forward PyTorch. Il normale session.run viene usato soltanto su CPU. Metriche, parità, provider e latenza confluiscono in BENCHMARK_RESULTS e nel file benchmark_results.json.

In [ ]:
GPU_ORT_PROVIDERS = {"CUDAExecutionProvider", "TensorrtExecutionProvider"}


def create_onnx_session(model_path):
    """Crea la sessione e blocca un fallback GPU -> CPU non desiderato."""
    session = ort.InferenceSession(str(model_path), providers=ORT_PROVIDERS)
    active_providers = session.get_providers()
    primary_provider = active_providers[0]

    # Il primo provider è quello prioritario. CPU può restare in fondo come
    # fallback per eventuali operatori non supportati dal provider principale.
    if PREFER_GPU and primary_provider not in GPU_ORT_PROVIDERS:
        raise RuntimeError(
            f"Era richiesta la GPU, ma il provider principale è {primary_provider}"
        )

    logger.info("ORT provider attivi: %s", active_providers)
    return session


def prepare_onnx_input(session, sample):
    """Converte il batch nel dtype dichiarato dall'input del modello ONNX."""
    input_type = session.get_inputs()[0].type
    numpy_dtypes = {
        "tensor(float)": np.float32,
        "tensor(float16)": np.float16,
    }
    if input_type not in numpy_dtypes:
        raise ValueError(f"Tipo input ONNX non gestito: {input_type}")
    return np.ascontiguousarray(sample.cpu().numpy(), dtype=numpy_dtypes[input_type])


@torch.inference_mode()
def compare_pytorch_and_onnx(model, session, sample, device):
    """Confronta logits e predizioni sullo stesso input FP32."""
    sample_numpy = prepare_onnx_input(session, sample)
    pytorch_logits = model(sample.to(device, non_blocking=True)).cpu().numpy()
    onnx_logits = session.run(
        [session.get_outputs()[0].name],
        {session.get_inputs()[0].name: sample_numpy},
    )[0]

    absolute_error = np.abs(pytorch_logits - onnx_logits)
    pytorch_predictions = np.argmax(pytorch_logits, axis=1)
    onnx_predictions = np.argmax(onnx_logits, axis=1)

    # allclose usa tolleranze strette adatte a due esecuzioni FP32.
    rtol, atol = 1e-4, 1e-5
    return {
        "output_shape": list(onnx_logits.shape),
        "max_abs_error": float(absolute_error.max()),
        "mean_abs_error": float(absolute_error.mean()),
        "prediction_agreement": float(
            np.mean(pytorch_predictions == onnx_predictions)
        ),
        "allclose_rtol": rtol,
        "allclose_atol": atol,
        "allclose": bool(
            np.allclose(pytorch_logits, onnx_logits, rtol=rtol, atol=atol)
        ),
    }


def evaluate_onnx(session, dataloader, split_name=None):
    """Calcola le stesse metriche della baseline PyTorch con ORT."""
    confusion_matrix = torch.zeros(
        (NUM_CLASSES, NUM_CLASSES), dtype=torch.float64
    )
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    for batch_number, batch in enumerate(dataloader, start=1):
        images = prepare_onnx_input(session, batch["image"])
        logits = session.run([output_name], {input_name: images})[0]
        predictions = torch.from_numpy(
            np.argmax(logits, axis=1).astype(np.int64, copy=False)
        )
        update_confusion_matrix(confusion_matrix, predictions, batch["mask"])

        if batch_number % 25 == 0 or batch_number == len(dataloader):
            logger.info("ONNX evaluation: %d/%d batch", batch_number, len(dataloader))

    metrics = metrics_from_confusion_matrix(confusion_matrix)
    metrics["split"] = split_name or EVALUATION_SPLIT
    metrics["samples"] = len(dataloader.dataset)
    return metrics


def measure_onnx_latency(session, sample):
    """Misura ORT con input pre-caricato sul device quando usa la GPU."""
    if sample.shape[0] != 1:
        raise ValueError("Il benchmark di latenza richiede batch size 1")

    sample_numpy = prepare_onnx_input(session, sample)
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    active_providers = session.get_providers()
    primary_provider = active_providers[0]

    if primary_provider in GPU_ORT_PROVIDERS:
        # OrtValue copia l'input sulla GPU prima del benchmark. I/O Binding
        # impedisce a ORT di riportare automaticamente l'output sulla CPU.
        io_binding = session.io_binding()
        input_ortvalue = ort.OrtValue.ortvalue_from_numpy(
            sample_numpy, "cuda", 0
        )
        io_binding.bind_ortvalue_input(input_name, input_ortvalue)
        io_binding.bind_output(output_name, "cuda", 0)

        def run_once():
            session.run_with_iobinding(io_binding)
            io_binding.synchronize_outputs()

        measurement_scope = "forward con input già sul device (I/O Binding)"
    else:
        def run_once():
            session.run([output_name], {input_name: sample_numpy})

        measurement_scope = "session.run su CPU"

    # Il warm-up non entra nelle statistiche: inizializza kernel e cache.
    for _ in range(LATENCY_WARMUP_RUNS):
        run_once()

    latencies_ms = []
    for _ in range(LATENCY_MEASURED_RUNS):
        start = time.perf_counter()
        run_once()
        latencies_ms.append((time.perf_counter() - start) * 1000.0)

    mean_ms = float(np.mean(latencies_ms))
    return {
        "provider": primary_provider,
        "provider_chain": active_providers,
        "batch_size": 1,
        "warmup_runs": LATENCY_WARMUP_RUNS,
        "measured_runs": LATENCY_MEASURED_RUNS,
        "measurement_scope": measurement_scope,
        "mean_ms": mean_ms,
        "median_ms": float(np.median(latencies_ms)),
        "p95_ms": float(np.percentile(latencies_ms, 95)),
        "images_per_second": 1000.0 / mean_ms,
    }


if RUN_ONNX_FP32_BENCHMARK:
    if "onnx_fp32_path" not in globals():
        raise RuntimeError("Esegui prima la cella di export ONNX FP32")
    if "evaluation_loader" not in globals():
        evaluation_dataset = build_dataset(EVALUATION_SPLIT, MAX_EVALUATION_SAMPLES)
        evaluation_loader = build_dataloader(evaluation_dataset)
    if "latency_sample" not in globals():
        latency_sample = next(iter(evaluation_loader))["image"]

    onnx_fp32_session = create_onnx_session(onnx_fp32_path)
    onnx_fp32_result = {
        "runtime": "onnxruntime",
        "precision": "fp32",
        "device": onnx_fp32_session.get_providers()[0],
        "providers": onnx_fp32_session.get_providers(),
        "artifact_key": "onnx_fp32",
        "numerical_parity_with_pytorch": compare_pytorch_and_onnx(
            model, onnx_fp32_session, latency_sample, TORCH_DEVICE
        ),
        "metrics": evaluate_onnx(onnx_fp32_session, evaluation_loader),
        "latency": measure_onnx_latency(onnx_fp32_session, latency_sample),
    }
    BENCHMARK_RESULTS["benchmarks"]["onnx_fp32_cuda"] = onnx_fp32_result
    results_path = save_benchmark_results()
    print(json.dumps(onnx_fp32_result, indent=2))
    print("Risultati cumulativi salvati in:", results_path)


## 11. Conversione e benchmark ONNX FP16

FP16 dimezza la larghezza dei valori floating point da 32 a 16 bit. Non è ancora PTQ INT8: non usa immagini di calibrazione, scale o zero-point. Convertiamo il grafo ONNX FP32 già verificato, poi ripetiamo gli stessi controlli prima di accettare il nuovo artefatto.

Con FP16_KEEP_IO_TYPES=True input e output pubblici restano FP32, mentre pesi e calcolo interno diventano FP16 dove supportato. Questo mantiene invariata l'interfaccia del modello. Impostando False anche input e output diventano FP16: può eliminare due Cast, ma obbliga il codice chiamante a fornire e ricevere tensori FP16.

Alcuni grafi con più `Resize` che condividono lo stesso tensore delle scale espongono due limiti del convertitore FP16 di ONNX Runtime: possono essere creati più `Cast` identici con lo stesso output e il `value_info` di una costante protetta può essere convertito a FP16 anche quando l'initializer corrispondente resta correttamente FP32. Prima del checker rimuoviamo soltanto duplicati byte-per-byte equivalenti e riallineiamo esclusivamente i tipi dei `value_info` associati agli initializer. Tutti gli altri tipi intermedi vengono preservati e qualsiasi collisione non equivalente o incompatibilità reale resta un errore bloccante.

In [ ]:
def onnx_element_type_name(value_info):
    """Restituisce FLOAT o FLOAT16 per un input/output ONNX."""
    element_type = value_info.type.tensor_type.elem_type
    return onnx.TensorProto.DataType.Name(element_type)


def deduplicate_equivalent_fp16_casts(model):
    """Rimuove solo Cast duplicati identici generati dal convertitore FP16."""
    producer_by_output = {}
    retained_nodes = []
    removed_cast_nodes = 0

    for node in model.graph.node:
        outputs = [name for name in node.output if name]
        duplicate_outputs = [name for name in outputs if name in producer_by_output]
        if duplicate_outputs:
            previous_nodes = [producer_by_output[name] for name in duplicate_outputs]
            node_bytes = node.SerializeToString()
            is_exact_duplicate_cast = (
                node.op_type == "Cast"
                and len(duplicate_outputs) == len(outputs)
                and all(
                    previous.op_type == "Cast"
                    and previous.SerializeToString() == node_bytes
                    for previous in previous_nodes
                )
            )
            if is_exact_duplicate_cast:
                removed_cast_nodes += 1
                continue

            raise RuntimeError(
                "Collisione SSA non equivalente dopo la conversione FP16: "
                f"nodo={node.name or node.op_type}, output={duplicate_outputs}. "
                "Il notebook non rinomina automaticamente tensori ambigui."
            )

        retained_nodes.append(node)
        for output_name in outputs:
            producer_by_output[output_name] = node

    del model.graph.node[:]
    model.graph.node.extend(retained_nodes)

    # Il convertitore può duplicare anche il metadata dello stesso Cast.
    # Conserviamo una sola copia se è identica e blocchiamo tipi conflittuali.
    value_info_by_name = {}
    retained_value_info = []
    removed_value_info = 0
    for value_info in model.graph.value_info:
        previous = value_info_by_name.get(value_info.name)
        if previous is None:
            value_info_by_name[value_info.name] = value_info
            retained_value_info.append(value_info)
        elif previous.SerializeToString() == value_info.SerializeToString():
            removed_value_info += 1
        else:
            raise RuntimeError(
                "Metadata ONNX conflittuale dopo la conversione FP16 per "
                f"il tensore {value_info.name}"
            )

    del model.graph.value_info[:]
    model.graph.value_info.extend(retained_value_info)
    return removed_cast_nodes, removed_value_info


def synchronize_initializer_value_info_types(model):
    """Allinea metadata e dtype reali delle costanti senza perdere le shape."""
    initializer_types = {
        initializer.name: initializer.data_type
        for initializer in model.graph.initializer
    }
    corrected = []
    for value_info in model.graph.value_info:
        initializer_type = initializer_types.get(value_info.name)
        tensor_type = value_info.type.tensor_type
        if (
            initializer_type is not None
            and tensor_type.elem_type != initializer_type
        ):
            corrected.append(
                {
                    "name": value_info.name,
                    "from": onnx.TensorProto.DataType.Name(tensor_type.elem_type),
                    "to": onnx.TensorProto.DataType.Name(initializer_type),
                }
            )
            tensor_type.elem_type = initializer_type
    return corrected


def convert_onnx_fp32_to_fp16(fp32_path):
    """Converte e valida un nuovo artefatto ONNX FP16."""
    fp32_path = Path(fp32_path)
    fp16_path = Path(ONNX_OUTPUT_DIR) / ONNX_FP16_FILENAME

    # 1. Carichiamo il grafo FP32 già validato. Non tocchiamo il checkpoint
    # PyTorch: la conversione avviene interamente nel formato ONNX.
    fp32_model = onnx.load(fp32_path)

    # 2. La funzione inclusa in ONNX Runtime converte tensori, pesi e nodi
    # compatibili da FLOAT a FLOAT16. Gli operatori non sicuri restano FP32
    # e vengono collegati automaticamente tramite nodi Cast.
    fp16_model = convert_float_to_float16(
        fp32_model,
        keep_io_types=FP16_KEEP_IO_TYPES,
        disable_shape_infer=False,
        force_fp16_initializers=False,
    )
    removed_cast_nodes, removed_value_info = deduplicate_equivalent_fp16_casts(
        fp16_model
    )
    if removed_cast_nodes or removed_value_info:
        logger.warning(
            "Workaround SSA FP16 applicato: rimossi %d Cast equivalenti e %d "
            "value_info equivalenti",
            removed_cast_nodes,
            removed_value_info,
        )
    corrected_initializer_types = synchronize_initializer_value_info_types(
        fp16_model
    )
    if corrected_initializer_types:
        logger.warning(
            "Metadata FP16 di initializer riallineati: %s",
            corrected_initializer_types,
        )

    # keep_io_types mantiene l'interfaccia FP32. disable_shape_infer=False
    # usa le shape note per convertire correttamente. Non forzare tutti gli
    # initializer evita di trasformare pesi usati da eventuali nodi FP32.

    # 3. La conversione può aggiungere i nuovi Cast in fondo alla lista dei
    # nodi. ONNXModel li riordina topologicamente prima di salvare, cioè mette
    # ogni nodo produttore prima dei nodi che ne consumano l'output.
    from onnxruntime.quantization.onnx_model import ONNXModel

    sortable_fp16_model = ONNXModel(fp16_model)
    sortable_fp16_model.save_model_to_file(
        str(fp16_path), use_external_data_format=False
    )
    # Il file resta separato: ONNX FP32 non viene sovrascritto e rimane il
    # riferimento per parità, metriche e conversioni successive.

    # 4. Il checker verifica schema, tipi e shape del nuovo grafo. Questo
    # controllo strutturale precede sempre inferenza e misure di accuratezza.
    checked_model = onnx.load(fp16_path)
    onnx.checker.check_model(checked_model, full_check=True)

    fp16_initializers = sum(
        initializer.data_type == onnx.TensorProto.FLOAT16
        for initializer in checked_model.graph.initializer
    )
    fp32_initializers = sum(
        initializer.data_type == onnx.TensorProto.FLOAT
        for initializer in checked_model.graph.initializer
    )
    cast_nodes = sum(node.op_type == "Cast" for node in checked_model.graph.node)
    if fp16_initializers == 0:
        raise RuntimeError("La conversione non ha prodotto pesi FP16")

    input_info = checked_model.graph.input[0]
    output_info = checked_model.graph.output[0]
    fp32_size = fp32_path.stat().st_size
    fp16_size = fp16_path.stat().st_size
    return fp16_path, {
        "path": str(fp16_path),
        "size_mib": fp16_size / (1024 ** 2),
        "size_reduction_percent_vs_fp32": 100.0 * (1.0 - fp16_size / fp32_size),
        "opset": next(
            item.version
            for item in checked_model.opset_import
            if item.domain in ("", "ai.onnx")
        ),
        "input_name": input_info.name,
        "input_type": onnx_element_type_name(input_info),
        "input_shape": onnx_tensor_shape(input_info),
        "output_name": output_info.name,
        "output_type": onnx_element_type_name(output_info),
        "output_shape": onnx_tensor_shape(output_info),
        "keep_io_types": FP16_KEEP_IO_TYPES,
        "fp16_initializers": fp16_initializers,
        "fp32_initializers": fp32_initializers,
        "cast_nodes": cast_nodes,
        "deduplicated_equivalent_cast_nodes": removed_cast_nodes,
        "deduplicated_equivalent_value_info": removed_value_info,
        "corrected_initializer_value_info_types": corrected_initializer_types,
        "onnx_checker": "passed",
    }


def compare_onnx_sessions(reference_session, candidate_session, sample):
    """Confronta ONNX FP16 con la baseline ONNX FP32 sullo stesso input."""
    reference_logits = reference_session.run(
        [reference_session.get_outputs()[0].name],
        {reference_session.get_inputs()[0].name: prepare_onnx_input(reference_session, sample)},
    )[0].astype(np.float32)
    candidate_logits = candidate_session.run(
        [candidate_session.get_outputs()[0].name],
        {candidate_session.get_inputs()[0].name: prepare_onnx_input(candidate_session, sample)},
    )[0].astype(np.float32)

    absolute_error = np.abs(reference_logits - candidate_logits)
    reference_predictions = np.argmax(reference_logits, axis=1)
    candidate_predictions = np.argmax(candidate_logits, axis=1)
    prediction_agreement = float(
        np.mean(reference_predictions == candidate_predictions)
    )

    # FP16 arrotonda più di FP32: allclose usa quindi tolleranze più ampie.
    # L'accordo delle classi e le metriche sull'intero validation set restano
    # i criteri principali, perché la segmentazione usa argmax sui logits.
    rtol, atol = 1e-2, 1e-2
    return {
        "reference": "onnx_fp32",
        "output_shape": list(candidate_logits.shape),
        "max_abs_error": float(absolute_error.max()),
        "mean_abs_error": float(absolute_error.mean()),
        "prediction_agreement": prediction_agreement,
        "allclose_rtol": rtol,
        "allclose_atol": atol,
        "allclose": bool(
            np.allclose(reference_logits, candidate_logits, rtol=rtol, atol=atol)
        ),
    }


if RUN_ONNX_FP16:
    if "onnx_fp32_path" not in globals():
        raise RuntimeError("Esegui prima la cella di export ONNX FP32")
    if "onnx_fp32_session" not in globals():
        onnx_fp32_session = create_onnx_session(onnx_fp32_path)
    if "evaluation_loader" not in globals():
        evaluation_dataset = build_dataset(EVALUATION_SPLIT, MAX_EVALUATION_SAMPLES)
        evaluation_loader = build_dataloader(evaluation_dataset)
    if "latency_sample" not in globals():
        latency_sample = next(iter(evaluation_loader))["image"]

    onnx_fp16_path, onnx_fp16_info = convert_onnx_fp32_to_fp16(onnx_fp32_path)
    BENCHMARK_RESULTS["artifacts"]["onnx_fp16"] = {
        "format": "onnx",
        "precision": "fp16",
        **onnx_fp16_info,
    }

    # Creare la sessione è anche un controllo operativo: il checker ONNX può
    # passare anche se il provider scelto non supporta qualche operatore FP16.
    onnx_fp16_session = create_onnx_session(onnx_fp16_path)
    onnx_fp16_result = {
        "runtime": "onnxruntime",
        "precision": "fp16",
        "device": onnx_fp16_session.get_providers()[0],
        "providers": onnx_fp16_session.get_providers(),
        "artifact_key": "onnx_fp16",
        "numerical_parity_with_onnx_fp32": compare_onnx_sessions(
            onnx_fp32_session, onnx_fp16_session, latency_sample
        ),
        "metrics": evaluate_onnx(onnx_fp16_session, evaluation_loader),
        "latency": measure_onnx_latency(onnx_fp16_session, latency_sample),
    }
    BENCHMARK_RESULTS["benchmarks"]["onnx_fp16_cuda"] = onnx_fp16_result
    results_path = save_benchmark_results()
    print(json.dumps({
        "artifact": onnx_fp16_info,
        "benchmark": onnx_fp16_result,
    }, indent=2))
    print("Risultati cumulativi salvati in:", results_path)


## 12. Preparazione e verifica della calibrazione INT8

Questa fase non quantizza ancora il modello. Riusa la selezione random/class-aware già definita, prepara l'interfaccia CalibrationDataReader richiesta da ONNX Runtime e verifica tutti i 500 input sul modello ONNX FP32. Le maschere restano nel DataLoader comune ma non vengono mai passate al calibratore.

Il manifesto salva fingerprint e indici del dataset originale: seed e strategia da soli non basterebbero a ricostruire lo stesso sottoinsieme se il dataset cambiasse ordine o revisione.

In [ ]:
from onnxruntime.quantization import CalibrationDataReader


class S5MarsCalibrationDataReader(CalibrationDataReader):
    """Espone al calibratore soltanto gli input immagine richiesti da ONNX."""

    def __init__(self, dataloader, input_name):
        if dataloader.batch_size != 1:
            raise ValueError("La calibrazione richiede batch size 1")
        self.dataset = dataloader.dataset
        self.input_name = input_name
        self.samples_yielded = 0
        self.start_index = 0
        self.end_index = len(self.dataset)
        self.next_index = self.start_index

    def __len__(self):
        # quantize_static usa questa lunghezza per suddividere la calibrazione.
        return len(self.dataset)

    def set_range(self, start_index, end_index):
        # Entropy conserva in RAM le attivazioni del range corrente. Intervalli
        # piccoli permettono di aggiornare gli istogrammi e liberarle a blocchi.
        start_index = int(start_index)
        end_index = int(end_index)
        if not 0 <= start_index < end_index <= len(self.dataset):
            raise ValueError(f"Intervallo calibrazione non valido: {start_index}:{end_index}")
        self.start_index = start_index
        self.end_index = end_index
        self.next_index = start_index
        self.samples_yielded = 0

    def get_next(self):
        # Restituiamo una sola immagine alla volta senza materializzare il subset.
        if self.next_index >= self.end_index:
            return None
        sample = self.dataset[self.next_index]
        self.next_index += 1

        # La calibrazione usa immagini FP32 già ridimensionate e normalizzate.
        # mask, index e source_index non entrano mai nel feed del modello.
        images = sample["image"].unsqueeze(0).numpy()
        images = np.ascontiguousarray(images, dtype=np.float32)
        self.samples_yielded += 1
        return {self.input_name: images}

    def rewind(self):
        # Ripristina il range completo per verifica o riuso del reader.
        self.start_index = 0
        self.end_index = len(self.dataset)
        self.next_index = self.start_index
        self.samples_yielded = 0


def verify_calibration_reader(reader, session, expected_samples, dry_run_samples):
    """Controlla tutti gli input e prova pochi forward, senza quantizzare."""
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    expected_shape = (1, 3, *IMAGE_SIZE)
    expected_output_shape = (1, NUM_CLASSES, *IMAGE_SIZE)
    sample_count = 0
    inference_runs = 0
    observed_min = float("inf")
    observed_max = float("-inf")

    reader.rewind()
    while True:
        model_inputs = reader.get_next()
        if model_inputs is None:
            break
        if set(model_inputs) != {input_name}:
            raise ValueError(f"Input del reader inattesi: {sorted(model_inputs)}")

        images = model_inputs[input_name]
        if images.dtype != np.float32:
            raise TypeError(f"Calibrazione attesa FP32, trovato {images.dtype}")
        if tuple(images.shape) != expected_shape:
            raise ValueError(f"Shape di calibrazione inattesa: {images.shape}")
        if not np.isfinite(images).all():
            raise ValueError("La calibrazione contiene NaN o valori infiniti")

        sample_count += images.shape[0]
        observed_min = min(observed_min, float(images.min()))
        observed_max = max(observed_max, float(images.max()))

        # Il dry-run usa solo pochi campioni: serve a verificare che gli input
        # siano realmente accettati dal modello FP32, non a misurare latenza.
        if inference_runs < dry_run_samples:
            logits = session.run([output_name], model_inputs)[0]
            if tuple(logits.shape) != expected_output_shape:
                raise ValueError(f"Output dry-run inatteso: {logits.shape}")
            if not np.isfinite(logits).all():
                raise ValueError("Il dry-run ha prodotto NaN o valori infiniti")
            inference_runs += 1

    if sample_count != expected_samples:
        raise ValueError(
            f"Attesi {expected_samples} campioni, il reader ne ha prodotti {sample_count}"
        )

    # Verifichiamo anche il contratto rewind, necessario per riusare il reader
    # nella futura chiamata di calibrazione senza ricreare il DataLoader.
    reader.rewind()
    rewind_ok = reader.get_next() is not None
    reader.rewind()
    if not rewind_ok:
        raise RuntimeError("CalibrationDataReader.rewind() non funziona")

    return {
        "status": "passed",
        "samples": sample_count,
        "input_name": input_name,
        "input_dtype": "float32",
        "input_shape": list(expected_shape),
        "all_inputs_finite": True,
        "normalized_value_min": observed_min,
        "normalized_value_max": observed_max,
        "dry_run_samples": inference_runs,
        "dry_run_output_shape": list(expected_output_shape),
        "all_dry_run_outputs_finite": True,
        "rewind_ok": rewind_ok,
    }


def summarize_class_presence(dataset):
    """Conta la presenza per immagine usando i metadati, non i pixel delle mask."""
    counts = {CLASS_NAMES[class_id]: 0 for class_id in range(NUM_CLASSES)}
    class_id_by_name = {name: class_id for class_id, name in CLASS_NAMES.items()}
    unknown_class_labels = set()

    for labels in dataset.dataset["class_labels"]:
        # Hugging Face può restituire le classi come nomi ("Bedrock")
        # oppure come ID (1). Normalizziamo entrambi nello stesso ID.
        image_class_ids = set()
        for value in labels:
            if isinstance(value, str) and value in class_id_by_name:
                image_class_ids.add(class_id_by_name[value])
                continue
            try:
                class_id = int(value)
            except (TypeError, ValueError):
                unknown_class_labels.add(str(value))
                continue
            if class_id in CLASS_NAMES:
                image_class_ids.add(class_id)
            else:
                unknown_class_labels.add(str(value))

        for class_id in image_class_ids:
            counts[CLASS_NAMES[class_id]] += 1

    missing_classes = [name for name, count in counts.items() if count == 0]
    return counts, missing_classes, sorted(unknown_class_labels)


if RUN_INT8_CALIBRATION_PREPARATION:
    if "onnx_fp32_path" not in globals():
        raise RuntimeError("Esegui prima la cella di export ONNX FP32")
    if "onnx_fp32_session" not in globals():
        onnx_fp32_session = create_onnx_session(onnx_fp32_path)

    # Riutilizziamo build_dataset, che applica la strategia configurata in
    # CALIBRATION_SELECTION: random oppure class_aware. Nessuna nuova selezione.
    calibration_dataset = build_dataset(
        CALIBRATION_SPLIT, MAX_CALIBRATION_SAMPLES, CALIBRATION_SELECTION
    )
    calibration_loader = build_dataloader(calibration_dataset, batch_size=1)
    calibration_reader = S5MarsCalibrationDataReader(
        calibration_loader, onnx_fp32_session.get_inputs()[0].name
    )
    calibration_verification = verify_calibration_reader(
        calibration_reader,
        onnx_fp32_session,
        expected_samples=MAX_CALIBRATION_SAMPLES,
        dry_run_samples=CALIBRATION_DRY_RUN_SAMPLES,
    )

    class_presence_counts, missing_classes, unknown_class_labels = (
        summarize_class_presence(calibration_dataset)
    )
    selected_source_indices = calibration_dataset.selected_source_indices
    indices_unique = len(set(selected_source_indices)) == len(selected_source_indices)
    indices_in_range = all(
        0 <= index < calibration_dataset.source_sample_count
        for index in selected_source_indices
    )
    if not indices_unique or not indices_in_range:
        raise ValueError("Gli indici del sottoinsieme di calibrazione non sono validi")
    if missing_classes:
        logger.warning("Classi assenti nelle 500 immagini: %s", missing_classes)
    if unknown_class_labels:
        logger.warning(
            "Classi sconosciute nei metadati: %s", unknown_class_labels
        )

    calibration_manifest = {
        "dataset_repo": REPO_ID,
        "split": CALIBRATION_SPLIT,
        "source_dataset_fingerprint": calibration_dataset.source_fingerprint,
        "source_sample_count": calibration_dataset.source_sample_count,
        "subset_fingerprint": calibration_dataset.subset_fingerprint,
        "selection": CALIBRATION_SELECTION,
        "seed": SUBSET_SEED,
        "sample_count": len(calibration_dataset),
        "batch_size": 1,
        "input_name": onnx_fp32_session.get_inputs()[0].name,
        "input_dtype": "float32",
        "input_shape": [1, 3, *IMAGE_SIZE],
        "normalization_mean": list(IMAGE_MEAN),
        "normalization_std": list(IMAGE_STD),
        "masks_used_for_calibration": False,
        "class_presence_unit": "images, not pixels",
        "class_presence_counts": class_presence_counts,
        "missing_classes": missing_classes,
        "unknown_class_labels": unknown_class_labels,
        "selected_indices_unique": indices_unique,
        "selected_indices_in_source_range": indices_in_range,
        "selected_source_indices": selected_source_indices,
    }

    # Il manifesto completo contiene tutti i 500 indici. Nell'output stampato
    # mostriamo solo inizio e fine per non produrre centinaia di righe.
    manifest_path = Path(ONNX_OUTPUT_DIR) / CALIBRATION_MANIFEST_FILENAME
    manifest_path.write_text(
        json.dumps(calibration_manifest, indent=2), encoding="utf-8"
    )
    available_providers = ort.get_available_providers()
    # get_available_providers() indica che il provider è installato, non
    # garantisce ancora che librerie CUDA/TensorRT e GPU siano utilizzabili.
    tensorrt_reported_available = (
        "TensorrtExecutionProvider" in available_providers
    )
    if not tensorrt_reported_available:
        logger.warning(
            "TensorRT EP non disponibile: la calibrazione è valida, ma il backend INT8 GPU va deciso"
        )

    BENCHMARK_RESULTS["artifacts"]["int8_calibration_manifest"] = {
        "format": "json",
        "path": str(manifest_path),
        "size_mib": manifest_path.stat().st_size / (1024 ** 2),
    }
    BENCHMARK_RESULTS["calibration"]["int8_preparation"] = {
        "manifest": calibration_manifest,
        "verification": calibration_verification,
        "available_ort_providers": available_providers,
        "tensorrt_reported_available": tensorrt_reported_available,
    }
    results_path = save_benchmark_results()

    calibration_output = {
        "manifest_path": str(manifest_path),
        "selection": CALIBRATION_SELECTION,
        "seed": SUBSET_SEED,
        "source_dataset_fingerprint": calibration_dataset.source_fingerprint,
        "subset_fingerprint": calibration_dataset.subset_fingerprint,
        "selected_indices_first_10": selected_source_indices[:10],
        "selected_indices_last_10": selected_source_indices[-10:],
        "class_presence_counts": class_presence_counts,
        "missing_classes": missing_classes,
        "unknown_class_labels": unknown_class_labels,
        "verification": calibration_verification,
        "available_ort_providers": available_providers,
        "tensorrt_reported_available": tensorrt_reported_available,
        "benchmark_results_path": str(results_path),
    }
    print(json.dumps(calibration_output, indent=2))


## 13. Creazione ONNX INT8 con PTQ statica

La quantizzazione statica esegue il modello FP32 sui 500 input e salva scale e zero-point nel nuovo grafo. Il checkpoint e gli artefatti FP32/FP16 non vengono modificati.

Scelte disponibili:

- **QDQ** inserisce coppie QuantizeLinear/DequantizeLinear ed è la scelta per TensorRT. **QOperator** sostituisce gli operatori con varianti quantizzate ed è rivolto soprattutto alla CPU.
- **MinMax** conserva l'intervallo osservato ed è il baseline più semplice. **Entropy** sceglie il range tramite istogrammi/KL divergence. **Percentile** elimina una piccola percentuale di valori estremi, aumentando la risoluzione ma rischiando saturazione.
- **Per-channel** assegna una scala a ogni canale di output dei pesi: costa pochi metadati e normalmente preserva meglio l'accuratezza. Le attivazioni restano per-tensor.
- **Simmetrica S8S8** usa INT8 signed e zero-point 0 per pesi e attivazioni; è il formato richiesto dal percorso GPU/TensorRT.
- **Conv soltanto** è il primo baseline conservativo. Ampliare gli operatori può aumentare l'accelerazione, ma anche errore numerico e coppie Q/DQ.

La configurazione versionata usa Entropy su 500 immagini random e richiede soltanto Conv in INT8. Lo scope deriva comunque da `INT8_OP_TYPES`: per esperimenti Conv+MatMul+Gemm la stessa cella registra sia gli operatori presenti nel grafo sorgente sia quelli effettivamente racchiusi da Q/DQ. LayerNormalization, Softmax, Add, Resize e gli altri nodi restano floating point, normalmente FP16 dentro TensorRT. Le coppie Q/DQ dedicate evitano condivisioni tra consumer non supportate da TensorRT.


In [ ]:
from onnxruntime.quantization import (
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
)


def count_onnx_op_types(model):
    counts = {}
    for node in model.graph.node:
        counts[node.op_type] = counts.get(node.op_type, 0) + 1
    return dict(sorted(counts.items()))


def count_qdq_wrapped_ops(model, configured_op_types):
    """Conta gli operatori configurati realmente racchiusi tra DQ e Q."""
    dequantized_tensors = {
        output
        for node in model.graph.node
        if node.op_type == "DequantizeLinear"
        for output in node.output
    }
    quantized_inputs = {
        node.input[0]
        for node in model.graph.node
        if node.op_type == "QuantizeLinear" and node.input
    }
    counts = {op_type: 0 for op_type in configured_op_types}
    for node in model.graph.node:
        if node.op_type not in counts:
            continue
        has_dequantized_input = any(
            name in dequantized_tensors for name in node.input
        )
        has_quantized_output = any(
            name in quantized_inputs for name in node.output
        )
        if has_dequantized_input and has_quantized_output:
            counts[node.op_type] += 1
    return counts


def audit_qdq_initializer_inputs(model):
    """Controlla sorgenti DQ e zero-point Q/DQ rispetto ai vincoli TensorRT."""
    initializers = {item.name: item for item in model.graph.initializer}
    source_type_counts = {}
    zero_point_type_counts = {}
    unsupported_nodes = []
    for node in model.graph.node:
        if node.op_type not in {"QuantizeLinear", "DequantizeLinear"} or not node.input:
            continue
        source_type = None
        if node.op_type == "DequantizeLinear":
            source = initializers.get(node.input[0])
            if source is not None:
                source_type = onnx.TensorProto.DataType.Name(source.data_type)
                source_type_counts[source_type] = source_type_counts.get(source_type, 0) + 1
        zero_point_type = None
        if len(node.input) >= 3 and node.input[2]:
            zero_point = initializers.get(node.input[2])
            if zero_point is not None:
                zero_point_type = onnx.TensorProto.DataType.Name(zero_point.data_type)
                zero_point_type_counts[zero_point_type] = (
                    zero_point_type_counts.get(zero_point_type, 0) + 1
                )
        if source_type in {"INT32", "UINT8"} or zero_point_type == "UINT8":
            unsupported_nodes.append(
                {
                    "node": node.name,
                    "op_type": node.op_type,
                    "source": node.input[0],
                    "source_type": source_type,
                    "zero_point_type": zero_point_type,
                }
            )
    return {
        "initializer_source_type_counts": dict(sorted(source_type_counts.items())),
        "zero_point_type_counts": dict(sorted(zero_point_type_counts.items())),
        "tensorrt_incompatible_nodes": unsupported_nodes,
    }


def decompose_conv_biases_for_tensorrt(fp32_path):
    """Sposta i bias Conv in Add broadcast, fuori dalle regioni QDQ INT8."""
    fp32_path = Path(fp32_path)
    model = onnx.load(fp32_path)
    initializers = {item.name: item for item in model.graph.initializer}
    used_names = set(initializers)
    used_names.update(node.name for node in model.graph.node if node.name)
    used_names.update(output for node in model.graph.node for output in node.output)

    def unique_name(base):
        candidate = base
        suffix = 1
        while candidate in used_names:
            candidate = f"{base}_{suffix}"
            suffix += 1
        used_names.add(candidate)
        return candidate

    rewritten_nodes = []
    decomposed = []
    unsupported = []
    for node in model.graph.node:
        if node.op_type != "Conv" or len(node.input) < 3 or not node.input[2]:
            rewritten_nodes.append(node)
            continue
        bias = initializers.get(node.input[2])
        weight = initializers.get(node.input[1])
        if bias is None or weight is None or len(weight.dims) < 3:
            unsupported.append(node.name or node.output[0])
            rewritten_nodes.append(node)
            continue
        bias_array = onnx.numpy_helper.to_array(bias)
        output_channels = int(weight.dims[0])
        spatial_rank = len(weight.dims) - 2
        if bias_array.ndim != 1 or bias_array.size != output_channels:
            unsupported.append(node.name or node.output[0])
            rewritten_nodes.append(node)
            continue

        original_output = node.output[0]
        conv_output = unique_name(f"{original_output}__without_bias")
        add_bias_name = unique_name(f"{node.input[2]}__broadcast")
        add_node_name = unique_name(f"{node.name or original_output}__bias_add")
        broadcast_shape = (1, output_channels) + (1,) * spatial_rank
        broadcast_bias = onnx.numpy_helper.from_array(
            bias_array.reshape(broadcast_shape), name=add_bias_name
        )
        model.graph.initializer.append(broadcast_bias)
        del node.input[2:]
        node.output[0] = conv_output
        rewritten_nodes.append(node)
        rewritten_nodes.append(
            onnx.helper.make_node(
                "Add",
                inputs=[conv_output, add_bias_name],
                outputs=[original_output],
                name=add_node_name,
            )
        )
        decomposed.append(node.name or original_output)

    if unsupported:
        raise RuntimeError(
            "Impossibile separare in modo sicuro il bias di alcune Conv: "
            f"{unsupported[:5]}"
        )
    del model.graph.node[:]
    model.graph.node.extend(rewritten_nodes)
    output_path = fp32_path.with_name(
        f"{fp32_path.stem}_trt_biasless_conv_source.onnx"
    )
    onnx.checker.check_model(model, full_check=True)
    onnx.save(model, output_path)
    return output_path, {
        "enabled": True,
        "path": str(output_path),
        "decomposed_conv_bias_count": len(decomposed),
        "decomposed_conv_nodes": decomposed,
        "added_float_add_nodes": len(decomposed),
        "onnx_checker": "passed",
    }


def verify_fp32_graph_rewrite(reference_path, candidate_path, data_reader):
    """Verifica su un input reale che la separazione Conv+bias sia equivalente."""
    providers = (
        ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if "CUDAExecutionProvider" in ort.get_available_providers()
        else ["CPUExecutionProvider"]
    )
    data_reader.rewind()
    sample = data_reader.get_next()
    data_reader.rewind()
    if sample is None:
        raise RuntimeError("CalibrationDataReader vuoto durante il gate del grafo PTQ")
    reference_session = ort.InferenceSession(str(reference_path), providers=providers)
    candidate_session = ort.InferenceSession(str(candidate_path), providers=providers)
    reference_output = reference_session.run(None, sample)[0]
    candidate_output = candidate_session.run(None, sample)[0]
    absolute_error = np.abs(reference_output - candidate_output)
    parity = {
        "max_abs_error": float(absolute_error.max()),
        "mean_abs_error": float(absolute_error.mean()),
        "prediction_agreement": float(
            np.mean(np.argmax(reference_output, axis=1) == np.argmax(candidate_output, axis=1))
        ),
        "allclose_rtol": 1e-5,
        "allclose_atol": 1e-5,
        "allclose": bool(
            np.allclose(reference_output, candidate_output, rtol=1e-5, atol=1e-5)
        ),
    }
    del reference_session, candidate_session
    if not parity["allclose"]:
        raise RuntimeError(
            "La separazione dei bias Conv non è numericamente equivalente: "
            f"{parity}"
        )
    return parity


def create_onnx_int8_static(fp32_path, calibration_data_reader):
    """Calibra il modello FP32 e crea un nuovo grafo ONNX QDQ INT8."""
    quant_formats = {
        "QDQ": QuantFormat.QDQ,
        "QOperator": QuantFormat.QOperator,
    }
    calibration_methods = {
        "MinMax": CalibrationMethod.MinMax,
        "Entropy": CalibrationMethod.Entropy,
        "Percentile": CalibrationMethod.Percentile,
    }
    quant_types = {
        "QInt8": QuantType.QInt8,
        "QUInt8": QuantType.QUInt8,
    }
    try:
        quant_format = quant_formats[INT8_QUANT_FORMAT]
        calibrate_method = calibration_methods[INT8_CALIBRATION_METHOD]
        activation_type = quant_types[INT8_ACTIVATION_TYPE]
        weight_type = quant_types[INT8_WEIGHT_TYPE]
    except KeyError as error:
        raise ValueError(f"Opzione PTQ non supportata: {error.args[0]}") from error

    if PREFER_GPU and (
        activation_type != QuantType.QInt8 or weight_type != QuantType.QInt8
    ):
        raise ValueError("Il percorso GPU richiede attivazioni e pesi QInt8")
    if PREFER_GPU and quant_format != QuantFormat.QDQ:
        raise ValueError("Il percorso TensorRT usa il formato QDQ")

    available = ort.get_available_providers()
    if PREFER_GPU:
        if "CUDAExecutionProvider" not in available:
            raise RuntimeError("CUDAExecutionProvider non disponibile per la calibrazione")
        calibration_providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    else:
        calibration_providers = ["CPUExecutionProvider"]

    fp32_path = Path(fp32_path)
    fp32_model = onnx.load(fp32_path)
    source_op_type_counts = count_onnx_op_types(fp32_model)
    configured_source_op_counts = {
        op_type: source_op_type_counts.get(op_type, 0)
        for op_type in INT8_OP_TYPES
    }
    requested_non_conv_ops = {
        op_type for op_type in INT8_OP_TYPES if op_type != "Conv"
    }
    absent_requested_ops = sorted(
        op_type
        for op_type, count in configured_source_op_counts.items()
        if count == 0
    )
    if absent_requested_ops:
        logger.warning(
            "Operatori INT8 richiesti ma assenti nel grafo: %s; si prosegue",
            absent_requested_ops,
        )
    if requested_non_conv_ops and not any(
        configured_source_op_counts.get(op_type, 0) > 0
        for op_type in requested_non_conv_ops
    ):
        raise RuntimeError(
            "Il grafo FP32 non contiene nessuno degli operatori non-Conv "
            f"configurati: {configured_source_op_counts}"
        )
    int8_path = Path(ONNX_OUTPUT_DIR) / ONNX_INT8_FILENAME
    quantization_source_path = fp32_path
    conv_bias_decomposition = {
        "enabled": False,
        "path": str(fp32_path),
        "decomposed_conv_bias_count": 0,
        "added_float_add_nodes": 0,
    }
    if INT8_DECOMPOSE_CONV_BIAS and "Conv" in INT8_OP_TYPES:
        if INT8_QUANTIZE_BIAS:
            raise ValueError(
                "INT8_DECOMPOSE_CONV_BIAS richiede INT8_QUANTIZE_BIAS=False"
            )
        quantization_source_path, conv_bias_decomposition = (
            decompose_conv_biases_for_tensorrt(fp32_path)
        )
        conv_bias_decomposition["numerical_parity_with_original"] = (
            verify_fp32_graph_rewrite(
                fp32_path, quantization_source_path, calibration_data_reader
            )
        )
    calibration_data_reader.rewind()
    calibration_extra_options = {
        # TensorRT usa quantizzazione simmetrica: zero-point uguale a 0.
        "ActivationSymmetric": INT8_SYMMETRIC,
        "WeightSymmetric": INT8_SYMMETRIC,
        "CalibTensorRangeSymmetric": INT8_SYMMETRIC,
        "DedicatedQDQPair": INT8_DEDICATED_QDQ_PAIR,
        # Evita DQ(INT32) sui bias: TensorRT accetta DQ espliciti INT8/FP8/
        # INT4, mentre i bias float vengono fusi/eseguiti nel fallback FP16.
        "QuantizeBias": INT8_QUANTIZE_BIAS,
    }
    if calibrate_method in {CalibrationMethod.Entropy, CalibrationMethod.Percentile}:
        chunk_size = int(INT8_CALIBRATION_CHUNK_SIZE)
        if chunk_size <= 0 or len(calibration_data_reader) % chunk_size != 0:
            raise ValueError(
                "INT8_CALIBRATION_CHUNK_SIZE deve essere positivo e dividere "
                f"esattamente {len(calibration_data_reader)}"
            )
        # Il nome ORT è storico: in ORT 1.26 questo stride suddivide anche
        # Entropy/Percentile. Ogni collect_data fonde gli istogrammi precedenti.
        calibration_extra_options["CalibStridedMinMax"] = chunk_size
    if calibrate_method == CalibrationMethod.Percentile:
        percentile = float(INT8_CALIBRATION_PERCENTILE)
        if not 0.0 < percentile <= 100.0:
            raise ValueError("INT8_CALIBRATION_PERCENTILE deve essere in (0, 100]")
        calibration_extra_options["CalibPercentile"] = percentile

    start = time.perf_counter()

    # Non aggiungiamo un pre-processing ONNX separato: il grafo esportato
    # ha già shape statiche verificate e quantize_static esegue shape inference.
    # L'eventuale avviso ORT sul pre-processing è quindi informativo.
    # quantize_static esegue il grafo FP32 sui 500 input, raccoglie i range
    # delle attivazioni e inserisce nel nuovo file scale, zero-point e Q/DQ.
    quantize_static(
        model_input=str(quantization_source_path),
        model_output=str(int8_path),
        calibration_data_reader=calibration_data_reader,
        quant_format=quant_format,
        activation_type=activation_type,
        weight_type=weight_type,
        calibrate_method=calibrate_method,
        per_channel=INT8_PER_CHANNEL,
        reduce_range=INT8_REDUCE_RANGE,
        op_types_to_quantize=list(INT8_OP_TYPES),
        use_external_data_format=False,
        calibration_providers=calibration_providers,
        extra_options=calibration_extra_options,
    )
    quantization_seconds = time.perf_counter() - start
    calibration_data_reader.rewind()

    # Il checker controlla schema, tipi e ordinamento del grafo prima di
    # provare a creare una sessione di inferenza nel prossimo step.
    int8_model = onnx.load(int8_path)
    onnx.checker.check_model(int8_model, full_check=True)
    qdq_initializer_audit = audit_qdq_initializer_inputs(int8_model)
    incompatible_qdq_nodes = qdq_initializer_audit[
        "tensorrt_incompatible_nodes"
    ]
    if incompatible_qdq_nodes:
        raise RuntimeError(
            "Il grafo QDQ contiene ancora DequantizeLinear incompatibili con "
            f"TensorRT: {incompatible_qdq_nodes[:5]}"
        )

    qdq_wrapped_op_counts = count_qdq_wrapped_ops(
        int8_model, INT8_OP_TYPES
    )
    present_non_conv_ops = {
        op_type
        for op_type, count in configured_source_op_counts.items()
        if op_type != "Conv" and count > 0
    }
    wrapped_non_conv_ops = {
        op_type
        for op_type, count in qdq_wrapped_op_counts.items()
        if op_type != "Conv" and count > 0
    }
    if not requested_non_conv_ops:
        non_conv_qdq_status = "not_requested"
    elif wrapped_non_conv_ops:
        non_conv_qdq_status = "wrapped"
    else:
        non_conv_qdq_status = "not_wrapped"
    if present_non_conv_ops and not wrapped_non_conv_ops:
        logger.warning(
            "Nessun operatore non-Conv presente nel grafo è stato "
            "racchiuso da Q/DQ; la quantizzazione prosegue. "
            "Presenti=%s, conteggi QDQ=%s",
            sorted(present_non_conv_ops),
            qdq_wrapped_op_counts,
        )

    initializers = {item.name: item for item in int8_model.graph.initializer}
    zero_point_names = {
        node.input[2]
        for node in int8_model.graph.node
        if node.op_type in {"QuantizeLinear", "DequantizeLinear"}
        and len(node.input) >= 3
        and node.input[2]
    }
    nonzero_zero_points = []
    for name in sorted(zero_point_names):
        initializer = initializers.get(name)
        if initializer is not None and np.any(onnx.numpy_helper.to_array(initializer) != 0):
            nonzero_zero_points.append(name)
    if INT8_SYMMETRIC and nonzero_zero_points:
        raise RuntimeError(
            f"Trovati zero-point non nulli nel modello simmetrico: {nonzero_zero_points[:5]}"
        )

    fp32_size = fp32_path.stat().st_size
    int8_size = int8_path.stat().st_size
    info = {
        "path": str(int8_path),
        "size_mib": int8_size / (1024 ** 2),
        "size_reduction_percent_vs_fp32": 100.0 * (1.0 - int8_size / fp32_size),
        "quantization_seconds": quantization_seconds,
        "quant_format": INT8_QUANT_FORMAT,
        "calibration_method": INT8_CALIBRATION_METHOD,
        "activation_type": INT8_ACTIVATION_TYPE,
        "weight_type": INT8_WEIGHT_TYPE,
        "symmetric": INT8_SYMMETRIC,
        "per_channel_weights": INT8_PER_CHANNEL,
        "reduce_range": INT8_REDUCE_RANGE,
        "dedicated_qdq_pair": INT8_DEDICATED_QDQ_PAIR,
        "quantize_bias": INT8_QUANTIZE_BIAS,
        "bias_runtime_precision": "int32_qdq" if INT8_QUANTIZE_BIAS else "float/fp16_fallback",
        "conv_bias_decomposition": conv_bias_decomposition,
        "op_types_requested": list(INT8_OP_TYPES),
        "source_op_type_counts": source_op_type_counts,
        "configured_source_op_counts": configured_source_op_counts,
        "requested_op_types_absent": absent_requested_ops,
        "qdq_wrapped_op_counts": qdq_wrapped_op_counts,
        "non_conv_ops_requested": sorted(requested_non_conv_ops),
        "non_conv_ops_present": sorted(present_non_conv_ops),
        "non_conv_ops_qdq_wrapped": sorted(wrapped_non_conv_ops),
        "non_conv_qdq_status": non_conv_qdq_status,
        "calibration_providers_requested": calibration_providers,
        "calibration_samples": MAX_CALIBRATION_SAMPLES,
        "calibration_selection": CALIBRATION_SELECTION,
        "calibration_chunk_size": calibration_extra_options.get("CalibStridedMinMax"),
        "calibration_percentile": calibration_extra_options.get("CalibPercentile"),
        "quantize_linear_nodes": sum(
            node.op_type == "QuantizeLinear" for node in int8_model.graph.node
        ),
        "dequantize_linear_nodes": sum(
            node.op_type == "DequantizeLinear" for node in int8_model.graph.node
        ),
        "int8_initializers": sum(
            item.data_type == onnx.TensorProto.INT8
            for item in int8_model.graph.initializer
        ),
        "uint8_initializers": sum(
            item.data_type == onnx.TensorProto.UINT8
            for item in int8_model.graph.initializer
        ),
        "nonzero_zero_points": nonzero_zero_points,
        "qdq_initializer_audit": qdq_initializer_audit,
        "input_type": onnx_element_type_name(int8_model.graph.input[0]),
        "input_shape": list(onnx_tensor_shape(int8_model.graph.input[0])),
        "output_type": onnx_element_type_name(int8_model.graph.output[0]),
        "output_shape": list(onnx_tensor_shape(int8_model.graph.output[0])),
        "onnx_checker": "passed",
    }
    if info["int8_initializers"] == 0 or info["quantize_linear_nodes"] == 0:
        raise RuntimeError("Il file creato non contiene quantizzazione INT8 QDQ")
    return int8_path, info


if RUN_ONNX_INT8_QUANTIZATION:
    if "onnx_fp32_path" not in globals():
        raise RuntimeError("Esegui prima l'export ONNX FP32")
    if "calibration_reader" not in globals():
        raise RuntimeError("Esegui prima la preparazione della calibrazione")

    onnx_int8_path, onnx_int8_info = create_onnx_int8_static(
        onnx_fp32_path, calibration_reader
    )
    onnx_int8_info["calibration_subset_fingerprint"] = (
        calibration_manifest["subset_fingerprint"]
    )
    BENCHMARK_RESULTS["artifacts"][INT8_EXPERIMENT_KEY] = {
        "format": "onnx",
        "precision": "int8",
        **onnx_int8_info,
    }
    BENCHMARK_RESULTS["calibration"]["int8_quantization"] = {
        key: onnx_int8_info[key]
        for key in (
            "calibration_method",
            "calibration_samples",
            "calibration_selection",
            "calibration_subset_fingerprint",
            "calibration_providers_requested",
        )
    }
    results_path = save_benchmark_results()
    print(json.dumps(onnx_int8_info, indent=2))
    print("Risultati cumulativi salvati in:", results_path)


## 14. Gate di esecuzione INT8 con TensorRT

Questo gate crea una sessione dedicata e svolge un solo forward. Non è ancora un benchmark: la prima esecuzione può includere la compilazione dell'engine TensorRT. Il profiling verifica quali provider abbiano realmente eseguito nodi; il semplice elenco di session.get_providers() non sarebbe sufficiente.

Il modello contiene già Q/DQ e le scale prodotte dalla calibrazione, quindi non viene passata alcuna calibration table a TensorRT. La cache dell'engine usa una cartella derivata dall'hash del modello, evitando di riutilizzare accidentalmente un engine creato per un artefatto diverso.


In [ ]:
def probe_tensorrt_int8(model_path, reference_session, sample):
    """Crea TensorRT, esegue un forward e controlla il provider via profiling."""
    import hashlib

    if "TensorrtExecutionProvider" not in ort.get_available_providers():
        raise RuntimeError("TensorrtExecutionProvider non è installato")

    model_path = Path(model_path)
    model_sha256 = hashlib.sha256(model_path.read_bytes()).hexdigest()

    # Un engine TensorRT dipende sia dal modello sia dalle opzioni di build.
    # Includere FP16/INT8, versioni e GPU nella firma evita di caricare un
    # engine costruito con una configurazione di precisione diversa.
    gpu_name = torch.cuda.get_device_name(TRT_DEVICE_ID)
    gpu_capability = torch.cuda.get_device_capability(TRT_DEVICE_ID)
    cache_signature = {
        "model_sha256": model_sha256,
        "ort_version": ort.__version__,
        "tensorrt_version": trt.__version__,
        "gpu_name": gpu_name,
        "gpu_compute_capability": list(gpu_capability),
        "device_id": TRT_DEVICE_ID,
        "trt_int8_enable": TRT_INT8_ENABLE,
        "trt_fp16_enable": TRT_FP16_ENABLE,
    }
    signature_json = json.dumps(cache_signature, sort_keys=True).encode("utf-8")
    signature_sha256 = hashlib.sha256(signature_json).hexdigest()
    cache_dir = (
        Path(TRT_ENGINE_CACHE_DIR)
        / f"{model_sha256[:16]}__{signature_sha256[:12]}"
    )
    cache_dir.mkdir(parents=True, exist_ok=True)
    timing_cache_dir = (
        Path(TRT_TIMING_CACHE_DIR)
        / f"cc_{gpu_capability[0]}{gpu_capability[1]}"
    )
    timing_cache_dir.mkdir(parents=True, exist_ok=True)
    cache_files_before = [path for path in cache_dir.rglob("*") if path.is_file()]

    # Il profiling è il controllo decisivo: session.get_providers() mostra
    # solo la priorità configurata, non garantisce che TensorRT esegua nodi.
    session_options = ort.SessionOptions()
    session_options.enable_profiling = True
    session_options.profile_file_prefix = str(
        Path(ONNX_OUTPUT_DIR) / "ort_trt_int8_probe"
    )

    trt_options = {
        "device_id": TRT_DEVICE_ID,
        "trt_int8_enable": TRT_INT8_ENABLE,
        "trt_fp16_enable": TRT_FP16_ENABLE,
        "trt_engine_cache_enable": True,
        "trt_engine_cache_path": str(cache_dir),
        # La timing cache riusa le misure dei kernel anche tra modelli INT8
        # simili; non sostituisce l'engine cache specifica dell'artefatto.
        "trt_timing_cache_enable": True,
        "trt_timing_cache_path": str(timing_cache_dir),
        # Nessuna calibration table: il modello QDQ contiene già le scale.
    }
    providers = [
        ("TensorrtExecutionProvider", trt_options),
        ("CUDAExecutionProvider", {"device_id": TRT_DEVICE_ID}),
        "CPUExecutionProvider",
    ]

    cache_state = "hit" if cache_files_before else "miss"
    print(
        f"TensorRT: creazione sessione, engine cache {cache_state} in {cache_dir}. "
        "Con cache miss la compilazione può richiedere diversi minuti...",
        flush=True,
    )
    creation_start = time.perf_counter()
    session = ort.InferenceSession(
        str(model_path), sess_options=session_options, providers=providers
    )
    session_creation_seconds = time.perf_counter() - creation_start
    print(
        f"TensorRT: sessione pronta in {session_creation_seconds:.1f} s; primo forward...",
        flush=True,
    )
    active_providers = session.get_providers()

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    model_inputs = {input_name: prepare_onnx_input(session, sample)}
    first_run_start = time.perf_counter()
    candidate_logits = session.run([output_name], model_inputs)[0]
    first_run_seconds = time.perf_counter() - first_run_start
    print(
        f"TensorRT: primo forward completato in {first_run_seconds:.3f} s; salvo il profilo...",
        flush=True,
    )
    profile_path = Path(session.end_profiling())

    expected_shape = (1, NUM_CLASSES, *IMAGE_SIZE)
    if tuple(candidate_logits.shape) != expected_shape:
        raise ValueError(f"Output TensorRT inatteso: {candidate_logits.shape}")
    if not np.isfinite(candidate_logits).all():
        raise ValueError("TensorRT ha prodotto NaN o valori infiniti")

    # Confronto su un solo input: serve a scoprire errori macroscopici.
    # Le metriche sull'intero validation set restano il controllo decisivo.
    reference_logits = reference_session.run(
        [reference_session.get_outputs()[0].name],
        {
            reference_session.get_inputs()[0].name:
                prepare_onnx_input(reference_session, sample)
        },
    )[0]
    absolute_error = np.abs(
        reference_logits.astype(np.float32) - candidate_logits.astype(np.float32)
    )
    prediction_agreement = float(
        np.mean(
            np.argmax(reference_logits, axis=1)
            == np.argmax(candidate_logits, axis=1)
        )
    )

    profile_events = json.loads(profile_path.read_text(encoding="utf-8"))
    provider_event_counts = {}
    for event in profile_events:
        provider = event.get("args", {}).get("provider")
        if provider:
            provider_event_counts[provider] = provider_event_counts.get(provider, 0) + 1

    tensorrt_event_count = provider_event_counts.get(
        "TensorrtExecutionProvider", 0
    )
    primary_is_tensorrt = bool(
        active_providers and active_providers[0] == "TensorrtExecutionProvider"
    )
    gate_passed = primary_is_tensorrt and tensorrt_event_count > 0
    cache_files_after = [path for path in cache_dir.rglob("*") if path.is_file()]

    result = {
        "status": "passed" if gate_passed else "failed",
        "model_path": str(model_path),
        "model_sha256": model_sha256,
        "engine_cache_signature": cache_signature,
        "active_providers": active_providers,
        "tensorrt_version": trt.__version__,
        "tensorrt_native_libraries": TENSORRT_LOADED_PATHS,
        "primary_provider_is_tensorrt": primary_is_tensorrt,
        "provider_event_counts": provider_event_counts,
        "tensorrt_executed_profile_events": tensorrt_event_count,
        "output_shape": list(candidate_logits.shape),
        "all_outputs_finite": True,
        "parity_with_onnx_fp32": {
            "max_abs_error": float(absolute_error.max()),
            "mean_abs_error": float(absolute_error.mean()),
            "prediction_agreement": prediction_agreement,
        },
        "session_creation_seconds": session_creation_seconds,
        "first_run_seconds_including_possible_engine_build": first_run_seconds,
        "engine_cache_path": str(cache_dir),
        "engine_cache_files_before": len(cache_files_before),
        "engine_cache_files_after": len(cache_files_after),
        "profile_path": str(profile_path),
        "provider_options": {
            "trt_int8_enable": TRT_INT8_ENABLE,
            "trt_fp16_enable": TRT_FP16_ENABLE,
            "calibration_table_supplied": False,
            "engine_cache_hit_before_session": bool(cache_files_before),
            "timing_cache_enabled": True,
        },
    }
    return session, result


if RUN_TRT_INT8_PROBE:
    if "onnx_int8_path" not in globals():
        raise RuntimeError("Esegui prima la creazione del modello ONNX INT8")
    if "onnx_fp32_session" not in globals():
        onnx_fp32_session = create_onnx_session(onnx_fp32_path)
    if "latency_sample" not in globals():
        if "evaluation_loader" not in globals():
            evaluation_dataset = build_dataset(
                EVALUATION_SPLIT, MAX_EVALUATION_SAMPLES
            )
            evaluation_loader = build_dataloader(evaluation_dataset)
        latency_sample = next(iter(evaluation_loader))["image"]

    onnx_int8_trt_session, onnx_int8_trt_probe = probe_tensorrt_int8(
        onnx_int8_path, onnx_fp32_session, latency_sample
    )
    BENCHMARK_RESULTS.setdefault("runtime_probes", {})[
        f"{INT8_EXPERIMENT_KEY}_tensorrt_probe"
    ] = onnx_int8_trt_probe
    BENCHMARK_RESULTS["artifacts"][f"{INT8_EXPERIMENT_KEY}_profile"] = {
        "format": "json",
        "path": onnx_int8_trt_probe["profile_path"],
    }
    results_path = save_benchmark_results()
    print(json.dumps(onnx_int8_trt_probe, indent=2))
    print("Risultati cumulativi salvati in:", results_path)


## 15. Metriche, latenza ed efficienza ONNX INT8 TensorRT

Questa fase viene eseguita soltanto dopo un probe TensorRT superato. Le metriche usano l'intero validation set e le stesse funzioni delle baseline. La latenza esclude creazione della sessione e compilazione dell'engine: usa 10 warm-up, 50 forward e I/O Binding con input/output sul device.

Il benchmark di efficienza è separato dalla latenza e dura almeno 30 secondi: NVML misura l'energia complessiva della GPU e campiona la memoria usata dal processo. `energy_per_image_mj` include il consumo idle del device durante la finestra; `peak_gpu_memory_mib` include modello, engine e buffer già residenti. Su una GPU condivisa l'energia può includere altri carichi e va considerata comparativa, non trasferibile direttamente al dispositivo edge.

Le misure includono sempre il forward completo, eventuali fallback e trasferimenti interni; non rappresentano il costo isolato dei soli kernel TensorRT. Il profiling salva i provider realmente usati e il flag `mixed_provider_execution`, quindi un run misto non può essere confuso con un engine interamente TensorRT.


In [ ]:
def build_onnx_gpu_runner(session, sample):
    """Prepara un forward con input/output residenti sulla GPU."""
    if sample.shape[0] != 1:
        raise ValueError("Il benchmark di efficienza richiede batch size 1")
    primary_provider = session.get_providers()[0]
    if primary_provider not in GPU_ORT_PROVIDERS:
        raise RuntimeError(
            f"Il benchmark NVML richiede un provider GPU, trovato {primary_provider}"
        )

    sample_numpy = prepare_onnx_input(session, sample)
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    io_binding = session.io_binding()
    input_ortvalue = ort.OrtValue.ortvalue_from_numpy(
        sample_numpy, "cuda", TRT_DEVICE_ID
    )
    io_binding.bind_ortvalue_input(input_name, input_ortvalue)
    io_binding.bind_output(output_name, "cuda", TRT_DEVICE_ID)

    def run_once():
        session.run_with_iobinding(io_binding)
        io_binding.synchronize_outputs()

    return run_once


class NvmlEfficiencyMonitor:
    """Misura energia device e picco memoria durante un loop sostenuto."""

    def __init__(self, device_id, sample_interval_seconds):
        self.device_id = int(device_id)
        self.sample_interval_seconds = float(sample_interval_seconds)
        if self.sample_interval_seconds <= 0:
            raise ValueError("NVML_SAMPLE_INTERVAL_SECONDS deve essere positivo")
        self.process_id = os.getpid()
        self.stop_event = threading.Event()
        self.samples = []
        self.sampling_error = None

    def _read_memory_mib(self):
        device_memory = pynvml.nvmlDeviceGetMemoryInfo(self.handle)
        try:
            processes = pynvml.nvmlDeviceGetComputeRunningProcesses(self.handle)
            for process in processes:
                if process.pid == self.process_id:
                    used = process.usedGpuMemory
                    # Alcune versioni di nvidia-ml-py non esportano una
                    # costante sentinella uniforme. Un valore reale deve
                    # comunque stare tra zero e la VRAM fisica totale.
                    if (
                        isinstance(used, (int, np.integer))
                        and 0 <= int(used) <= int(device_memory.total)
                    ):
                        return int(used) / (1024 ** 2), "process"
        except (AttributeError, pynvml.NVMLError):
            pass
        return device_memory.used / (1024 ** 2), "device_total_fallback"

    def _sample_once(self):
        timestamp = time.perf_counter()
        power_w = pynvml.nvmlDeviceGetPowerUsage(self.handle) / 1000.0
        memory_mib, memory_scope = self._read_memory_mib()
        self.samples.append((timestamp, power_w, memory_mib, memory_scope))

    def _sample_loop(self):
        try:
            while not self.stop_event.wait(self.sample_interval_seconds):
                self._sample_once()
        except Exception as error:
            self.sampling_error = repr(error)
            self.stop_event.set()

    def start(self):
        pynvml.nvmlInit()
        self.handle = pynvml.nvmlDeviceGetHandleByIndex(self.device_id)
        self.samples = []
        self.sampling_error = None
        self.stop_event.clear()
        try:
            self.energy_start_mj = pynvml.nvmlDeviceGetTotalEnergyConsumption(
                self.handle
            )
            self.total_energy_counter_available = True
        except (AttributeError, pynvml.NVMLError):
            self.energy_start_mj = None
            self.total_energy_counter_available = False
        self._sample_once()
        self.thread = threading.Thread(target=self._sample_loop, daemon=True)
        self.thread.start()

    def stop(self):
        energy_end_mj = None
        if self.total_energy_counter_available:
            energy_end_mj = pynvml.nvmlDeviceGetTotalEnergyConsumption(
                self.handle
            )
        self.stop_event.set()
        self.thread.join(timeout=max(1.0, 5 * self.sample_interval_seconds))
        self._sample_once()
        if self.sampling_error is not None:
            raise RuntimeError(f"Campionamento NVML fallito: {self.sampling_error}")
        if not self.samples:
            raise RuntimeError("NVML non ha prodotto campioni")

        if energy_end_mj is not None:
            total_energy_j = (energy_end_mj - self.energy_start_mj) / 1000.0
            energy_source = "nvml_total_energy_counter"
        else:
            total_energy_j = sum(
                0.5 * (left[1] + right[1]) * (right[0] - left[0])
                for left, right in zip(self.samples, self.samples[1:])
            )
            energy_source = "integrated_nvml_power_samples"
        if total_energy_j <= 0:
            raise RuntimeError(f"Energia NVML non valida: {total_energy_j}")

        return {
            "total_energy_j": float(total_energy_j),
            "energy_source": energy_source,
            "energy_scope": "whole GPU device, idle not subtracted",
            "average_sampled_power_w": float(
                np.mean([sample[1] for sample in self.samples])
            ),
            "peak_gpu_memory_mib": float(
                max(sample[2] for sample in self.samples)
            ),
            "gpu_memory_scope": (
                "process"
                if all(sample[3] == "process" for sample in self.samples)
                else "device_total_fallback"
            ),
            "nvml_samples": len(self.samples),
            "nvml_sample_interval_seconds": self.sample_interval_seconds,
        }


def measure_onnx_efficiency(session, sample):
    """Restituisce energia per immagine e memoria GPU di picco."""
    min_seconds = float(EFFICIENCY_MIN_SECONDS)
    min_runs = int(EFFICIENCY_MIN_RUNS)
    if min_seconds <= 0 or min_runs <= 0:
        raise ValueError("Durata e numero minimo di run devono essere positivi")

    run_once = build_onnx_gpu_runner(session, sample)
    for _ in range(EFFICIENCY_WARMUP_RUNS):
        run_once()

    monitor = NvmlEfficiencyMonitor(
        TRT_DEVICE_ID, NVML_SAMPLE_INTERVAL_SECONDS
    )
    monitor.start()
    measured_runs = 0
    start = time.perf_counter()
    try:
        while measured_runs < min_runs or time.perf_counter() - start < min_seconds:
            run_once()
            measured_runs += 1
        elapsed_seconds = time.perf_counter() - start
    finally:
        resource_metrics = monitor.stop()

    energy_per_image_mj = (
        resource_metrics["total_energy_j"] * 1000.0 / measured_runs
    )
    return {
        "average_sampled_power_w": resource_metrics[
            "average_sampled_power_w"
        ],
        "energy_per_image_mj": float(energy_per_image_mj),
        "images_per_joule": float(1000.0 / energy_per_image_mj),
        "peak_gpu_memory_mib": resource_metrics[
            "peak_gpu_memory_mib"
        ],
    }


if RUN_ONNX_INT8_BENCHMARK:
    if "onnx_int8_trt_session" not in globals():
        raise RuntimeError("Esegui prima il gate TensorRT della sezione 14")
    if onnx_int8_trt_probe.get("status") != "passed":
        raise RuntimeError("Il gate TensorRT non è stato superato")
    if "evaluation_loader" not in globals():
        evaluation_dataset = build_dataset(
            EVALUATION_SPLIT, MAX_EVALUATION_SAMPLES
        )
        evaluation_loader = build_dataloader(evaluation_dataset)
    if "latency_sample" not in globals():
        latency_sample = next(iter(evaluation_loader))["image"]

    # Riutilizziamo la sessione già validata: l'engine è costruito/caricato
    # e il costo di inizializzazione non entra né nelle metriche né nella latenza.
    int8_metrics = evaluate_onnx(onnx_int8_trt_session, evaluation_loader)
    int8_latency = measure_onnx_latency(
        onnx_int8_trt_session, latency_sample
    )

    # La memoria NVML è per processo: prima della misura lasciamo residente
    # una sola sessione ONNX. Il modello PyTorch non serve più sulla GPU.
    if "model" in globals():
        model.to("cpu")
    for session_name in ("onnx_fp32_session", "onnx_fp16_session"):
        globals().pop(session_name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    int8_efficiency = measure_onnx_efficiency(
        onnx_int8_trt_session, latency_sample
    )
    onnx_int8_result = {
        "runtime": "onnxruntime",
        "precision": "int8_qdq",
        "device": onnx_int8_trt_session.get_providers()[0],
        "providers": onnx_int8_trt_session.get_providers(),
        "artifact_key": INT8_EXPERIMENT_KEY,
        "calibration_method": INT8_CALIBRATION_METHOD,
        "calibration_selection": CALIBRATION_SELECTION,
        "probe_status": onnx_int8_trt_probe["status"],
        "provider_event_counts_from_probe": onnx_int8_trt_probe[
            "provider_event_counts"
        ],
        "mixed_provider_execution": len(
            onnx_int8_trt_probe["provider_event_counts"]
        ) > 1,
        "numerical_parity_with_onnx_fp32": onnx_int8_trt_probe[
            "parity_with_onnx_fp32"
        ],
        "metrics": int8_metrics,
        "latency": int8_latency,
        "efficiency": int8_efficiency,
        "latency_excludes_session_and_engine_build": True,
        "tensorrt_provider_options": onnx_int8_trt_probe["provider_options"],
    }
    BENCHMARK_RESULTS["benchmarks"][
        f"{INT8_EXPERIMENT_KEY}_tensorrt"
    ] = onnx_int8_result

    # Misuriamo FP32 e FP16 con una sessione alla volta, usando lo stesso
    # loop sostenuto e le stesse quattro metriche dell'esperimento INT8.
    globals().pop("onnx_int8_trt_session", None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    efficiency_variants = (
        ("onnx_fp32_cuda", onnx_fp32_path),
        ("onnx_fp16_cuda", onnx_fp16_path),
    )
    for benchmark_key, model_path in efficiency_variants:
        if benchmark_key not in BENCHMARK_RESULTS["benchmarks"]:
            logger.warning(
                "Benchmark %s assente: efficienza non misurata",
                benchmark_key,
            )
            continue
        efficiency_session = create_onnx_session(model_path)
        BENCHMARK_RESULTS["benchmarks"][benchmark_key]["efficiency"] = (
            measure_onnx_efficiency(efficiency_session, latency_sample)
        )
        del efficiency_session
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    results_path = save_benchmark_results()
    print(json.dumps(onnx_int8_result, indent=2))
    print("Risultati cumulativi salvati in:", results_path)


## 16. Verifica finale sul test set

Questa cella è un gate finale separato. Rimane disattivata durante la scelta di modello, calibrazione e operatori sulla validation. Quando `RUN_FINAL_TEST_EVALUATION=True`, calcola soltanto pixel accuracy, mIoU e IoU per classe sull'intero test set per PyTorch FP32, ONNX FP32, ONNX FP16 e INT8. Non ripete latenza o misure energetiche.

In [ ]:
if RUN_FINAL_TEST_EVALUATION:
    if "model" not in globals():
        raise RuntimeError("Carica prima modello e checkpoint")

    test_dataset = build_dataset(TEST_SPLIT, MAX_TEST_SAMPLES)
    test_loader = build_dataloader(test_dataset)
    test_sample = next(iter(test_loader))["image"]
    final_test_metrics = {
        "protocol": (
            "hold-out finale; non usare per scegliere configurazione o calibrazione"
        ),
        "split": TEST_SPLIT,
        "samples": len(test_dataset),
    }

    # Una variante alla volta limita la memoria residente; non misuriamo
    # qui memoria o energia, quindi la ricreazione delle sessioni è esclusa.
    model.to(TORCH_DEVICE)
    final_test_metrics["pytorch_fp32_cuda"] = evaluate_pytorch_fp32(
        model, test_loader, TORCH_DEVICE, split_name=TEST_SPLIT
    )
    model.to("cpu")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    test_fp32_session = create_onnx_session(onnx_fp32_path)
    final_test_metrics["onnx_fp32_cuda"] = evaluate_onnx(
        test_fp32_session, test_loader, split_name=TEST_SPLIT
    )
    test_fp16_session = create_onnx_session(onnx_fp16_path)
    final_test_metrics["onnx_fp16_cuda"] = evaluate_onnx(
        test_fp16_session, test_loader, split_name=TEST_SPLIT
    )
    del test_fp16_session
    gc.collect()

    test_int8_session, test_int8_probe = probe_tensorrt_int8(
        onnx_int8_path, test_fp32_session, test_sample
    )
    if test_int8_probe["status"] != "passed":
        raise RuntimeError("Il gate TensorRT INT8 sul test non è stato superato")
    final_test_metrics[f"{INT8_EXPERIMENT_KEY}_tensorrt"] = evaluate_onnx(
        test_int8_session, test_loader, split_name=TEST_SPLIT
    )
    BENCHMARK_RESULTS["test_metrics"] = final_test_metrics
    results_path = save_benchmark_results()
    print(json.dumps(final_test_metrics, indent=2))
    print("Risultati test salvati in:", results_path)
else:
    print(
        "Test non eseguito: attiva RUN_FINAL_TEST_EVALUATION soltanto "
        "dopo la selezione finale sulla validation."
    )


## 17. Riepilogo rapido dei modelli testati

Questa cella non esegue altra inferenza: legge soltanto BENCHMARK_RESULTS e stampa una riga per ogni variante già misurata. I delta sono calcolati rispetto a PyTorch FP32; un delta di latenza negativo indica un modello più veloce.

In [ ]:
# False stampa soltanto la tabella compatta. Imposta True quando vuoi anche
# il dizionario completo con IoU per classe, provider e metadati dei file.
PRINT_FULL_RESULTS_JSON = False


def print_benchmark_summary(results, reference_key="pytorch_fp32_cuda"):
    """Stampa il confronto essenziale senza dipendere da pandas."""
    benchmarks = results.get("benchmarks", {})
    if not benchmarks:
        raise RuntimeError("Nessun benchmark disponibile")

    # Manteniamo un ordine stabile; eventuali varianti future vengono aggiunte
    # in fondo automaticamente, senza modificare questa funzione.
    preferred_order = [
        "pytorch_fp32_cuda",
        "onnx_fp32_cuda",
        "onnx_fp16_cuda",
        "onnx_int8_tensorrt",
    ]
    ordered_keys = [key for key in preferred_order if key in benchmarks]
    ordered_keys.extend(sorted(key for key in benchmarks if key not in ordered_keys))

    reference = benchmarks.get(reference_key)
    if reference is None:
        reference_key = ordered_keys[0]
        reference = benchmarks[reference_key]
    reference_miou = reference["metrics"]["miou"]
    reference_mean_ms = reference["latency"]["mean_ms"]

    header = (
        f"{'variant':<55} {'pixel acc':>10} {'mIoU':>10} {'ΔmIoU':>11} "
        f"{'mean ms':>10} {'Δlatency':>11} {'p95 ms':>10} {'img/s':>10} "
        f"{'mJ/img':>10} {'peak MiB':>10}"
    )
    print("Riferimento:", reference_key)
    print(header)
    print("-" * len(header))

    for key in ordered_keys:
        benchmark = benchmarks[key]
        pixel_accuracy = benchmark["metrics"]["pixel_accuracy"]
        miou = benchmark["metrics"]["miou"]
        latency = benchmark["latency"]
        mean_ms = latency["mean_ms"]
        efficiency = benchmark.get("efficiency", {})
        energy_per_image = efficiency.get("energy_per_image_mj")
        peak_gpu_memory = efficiency.get("peak_gpu_memory_mib")
        energy_text = (
            f"{energy_per_image:.3f}" if energy_per_image is not None else "n.d."
        )
        memory_text = (
            f"{peak_gpu_memory:.1f}" if peak_gpu_memory is not None else "n.d."
        )
        delta_miou = miou - reference_miou
        delta_latency_percent = 100.0 * (mean_ms / reference_mean_ms - 1.0)
        print(
            f"{key:<55} {pixel_accuracy:>10.6f} {miou:>10.6f} "
            f"{delta_miou:>+11.6f} "
            f"{mean_ms:>10.3f} {delta_latency_percent:>+10.1f}% "
            f"{latency['p95_ms']:>10.3f} {latency['images_per_second']:>10.2f} "
            f"{energy_text:>10} {memory_text:>10}"
        )


print_benchmark_summary(BENCHMARK_RESULTS)
print("\nRisultati completi:", save_benchmark_results())
if PRINT_FULL_RESULTS_JSON:
    print(json.dumps(BENCHMARK_RESULTS, indent=2))
